# Multimodal Dynamic Knowledge Graph Pipeline for Urban Scene Understanding

## Overview
This notebook implements an end-to-end pipeline that processes dashcam/surveillance video 
to build a **Dynamic Knowledge Graph (DKG)** of the scene, detect events, and generate 
natural language narratives.

## Pipeline Architecture
| Module | Name | Role |
|--------|------|------|
| M0 | Preprocessor | Audio extraction + YAMNet tokenisation |
| M1 | Perception | YOLOv11x object detection + ByteTrack |
| M2 | Tracking | Velocity, acceleration, direction features |
| M3 | Relation Inference | Spatial + audio-visual triplet generation |
| M4 | Dynamic Knowledge Graph | Temporal Bayesian graph construction |
| M5 | Event Engine | Rule-based event detection (16 event types) |
| M6 | Summarizer | Phi-3 Mini narrative generation |
| M7 | Visualizer | DKG plots + annotated video |
| M8 | Action & Deployment | Priority dispatch + incident deduplication |

## Dataset
TAU , MAVAD and Few test videos used for evaluation.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 1: Install Dependencies                        ║
# ╚══════════════════════════════════════════════════════════╝

# ── Vision ────────────────────────────────────────────────
!pip install -q ultralytics==8.3.0
!pip install -q supervision==0.20.0
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# ── Audio ─────────────────────────────────────────────────
!pip install -q "tensorflow==2.15.0" tensorflow-hub
!pip install -q librosa soundfile   
!apt-get install -qq ffmpeg

# ── NLP / Summarization ───────────────────────────────────
# transformers: Phi-3 Mini for narrative generation
# bitsandbytes: enables 4-bit quantization in M6 to reduce VRAM usage
!pip install -q "transformers==4.44.0" accelerate bitsandbytes

# ── Graph & Visualization ─────────────────────────────────
!pip install -q networkx==3.3
!pip install -q scipy matplotlib pyvis

# ── Evaluation ────────────────────────────────────────────
!pip install -q bert-score

print("All dependencies installed.")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 2: Configuration                               ║
# ╚══════════════════════════════════════════════════════════╝

import os
import cv2

VIDEO_PATH  = "/kaggle/input/datasets/dharshini25v/batch-processing/street_pedestrian-stockholm-155-4709.mp4"
WORKING_DIR = "/kaggle/working"
YOLO_MODEL  = "yolo11x.pt"

# ── Auto-detect frame count and FPS ───────────────────────
_cap        = cv2.VideoCapture(VIDEO_PATH)
MAX_FRAMES  = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))   # full video
# Set MAX_FRAMES = 300 for quick testing on long videos
_cap.release()

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"[Config] Video     : {os.path.basename(VIDEO_PATH)}")
print(f"[Config] WorkingDir: {WORKING_DIR}")
print(f"[Config] MaxFrames : {MAX_FRAMES}  (full video)")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 3: Utility Functions                           ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict


def reencode_video(input_path: str, working_dir: str) -> str:
    """Re-encode video to H.264 if needed. Returns path to usable video."""
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-select_streams", "v:0",
         "-show_entries", "stream=codec_name", "-of", "csv=p=0", input_path],
        capture_output=True, text=True
    )
    codec = probe.stdout.strip().lower()
    if codec in ("h264", "avc"):
        print(f"[Encode] Already H.264 — skipping re-encode.")
        return input_path

    out_name = Path(input_path).stem + "_h264.mp4"
    out_path = os.path.join(working_dir, out_name)
    cmd = ["ffmpeg", "-y", "-i", input_path,
           "-c:v", "libx264", "-preset", "fast", "-crf", "18",
           "-pix_fmt", "yuv420p",
           "-c:a", "aac", "-b:a", "128k", out_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[Encode] Warning: ffmpeg error:\n{result.stderr[-400:]}")
        return input_path
    print(f"[Encode] Re-encoded → {out_path}")
    return out_path


def get_bbox(row) -> tuple:
    """Robustly extract (x1, y1, x2, y2) from a detection row."""
    try:
        if "x1" in row and pd.notna(row["x1"]):
            return int(row["x1"]), int(row["y1"]), int(row["x2"]), int(row["y2"])
        raw = row["bbox"]
        if isinstance(raw, str):
            raw = raw.strip("[()]").split(",")
        return int(float(raw[0])), int(float(raw[1])), int(float(raw[2])), int(float(raw[3]))
    except Exception:
        return 0, 0, 0, 0


def build_frame_index(df: pd.DataFrame) -> Dict[int, pd.DataFrame]:
    """Pre-index a detections dataframe by frame_idx for fast per-frame lookup."""
    df = df.copy()
    df["frame_idx"] = df["frame_idx"].astype(int)
    return {int(fidx): grp for fidx, grp in df.groupby("frame_idx")}


def get_stable_class(df: pd.DataFrame) -> Dict[int, str]:
    """Return a dict mapping track_id → most common class_name (prevents label flicker)."""
    return (
        df.groupby("track_id")["class_name"]
          .agg(lambda s: s.mode().iloc[0])
          .to_dict()
    )


def iou(box1, box2) -> float:
    """Compute IoU between two (x1,y1,x2,y2) boxes."""
    ix1 = max(box1[0], box2[0]); iy1 = max(box1[1], box2[1])
    ix2 = min(box1[2], box2[2]); iy2 = min(box1[3], box2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)


def deduplicate_and_reidentify(
    detections_df: pd.DataFrame,
    fps: float,
    max_jump_px: float = 200.0,
    min_gap_frames: int = 3,
    max_gap_frames: int = 30,
    iou_thresh: float = 0.3,
) -> pd.DataFrame:
    """
    Phase 1: Split tracks that jump too far between frames (centroid-jump split).
    Phase 2: Re-identify tracks lost during occlusion by matching last-known
             bbox + class against newly appeared tracks within max_gap_frames.
    Known limitation: id_map does not resolve chained merges (A→B→C won't merge A→C).
    """
    if detections_df.empty:
        return detections_df

    df = detections_df.copy()
    df["frame_idx"] = df["frame_idx"].astype(int)
    df = df.sort_values(["track_id", "frame_idx"]).reset_index(drop=True)

    # ── Phase 1: centroid-jump split ──────────────────────
    next_new_id = int(df["track_id"].max()) + 1
    total_splits = 0
    changed = True
    while changed:
        changed = False
        for tid in df["track_id"].unique():
            rows  = df[df["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < 2: continue
            idxs  = rows.index.tolist()
            fidxs = rows["frame_idx"].values
            cxs   = rows["bbox_x"].values if "bbox_x" in rows else ((rows["x1"]+rows["x2"])/2).values
            cys   = rows["bbox_y"].values if "bbox_y" in rows else ((rows["y1"]+rows["y2"])/2).values
            for i in range(1, len(idxs)):
                gap  = int(fidxs[i]) - int(fidxs[i-1])
                if gap < min_gap_frames: continue
                disp = float(np.hypot(cxs[i]-cxs[i-1], cys[i]-cys[i-1]))
                if disp > max_jump_px:
                    df.loc[idxs[i:], "track_id"] = next_new_id
                    next_new_id += 1; total_splits += 1; changed = True; break

    # ── Phase 2: re-identification across gaps ─────────────
    track_ends   = {}
    track_starts = {}
    for tid in df["track_id"].unique():
        rows  = df[df["track_id"] == tid].sort_values("frame_idx")
        last  = rows.iloc[-1];  first = rows.iloc[0]
        box_l = (float(last["x1"]),  float(last["y1"]),  float(last["x2"]),  float(last["y2"]))
        box_f = (float(first["x1"]), float(first["y1"]), float(first["x2"]), float(first["y2"]))
        track_ends[tid]   = (int(last["frame_idx"]),  str(last["class_name"]),  box_l)
        track_starts[tid] = (int(first["frame_idx"]), str(first["class_name"]), box_f)

    merged = 0
    id_map = {}
    for tid_new, (fstart, cls_new, box_new) in track_starts.items():
        for tid_old, (fend, cls_old, box_old) in track_ends.items():
            if tid_new == tid_old: continue
            gap = fstart - fend
            if not (min_gap_frames <= gap <= max_gap_frames): continue
            if cls_new != cls_old: continue
            if iou(box_old, box_new) >= iou_thresh:
                id_map[tid_new] = tid_old
                merged += 1
                break

    for tid_new, tid_old in id_map.items():
        df.loc[df["track_id"] == tid_new, "track_id"] = tid_old

    print(f"[Dedup+ReID] Phase1 splits={total_splits} | Phase2 merges={merged} | "
          f"final unique tracks={df['track_id'].nunique()}")
    return df


print("[Utils] Utility functions loaded.")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 4: EDA — Video Preview                         ║
# ╚══════════════════════════════════════════════════════════╝

import cv2, numpy as np, matplotlib.pyplot as plt

cap   = cv2.VideoCapture(VIDEO_PATH)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_v = cap.get(cv2.CAP_PROP_FPS)
sample_frames = []

# Sample 8 frames evenly across the full processing range
for tgt in np.linspace(0, min(total - 1, MAX_FRAMES - 1), 8, dtype=int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(tgt))
    ret, frame = cap.read()
    if ret:
        sample_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

fig, axes = plt.subplots(2, 4, figsize=(18, 6))
for ax, frm in zip(axes.flat, sample_frames):
    ax.imshow(frm); ax.axis("off")
fig.suptitle(f"{os.path.basename(VIDEO_PATH)}  |  {total} frames @ {fps_v:.1f} fps", fontsize=12)
plt.tight_layout(); plt.show()
print(f"[Preview] {total} frames, {fps_v:.1f} fps")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 5: Re-encode Video                             ║
# ╚══════════════════════════════════════════════════════════╝

OUTPUT_PATH = reencode_video(VIDEO_PATH, WORKING_DIR)
print(f"[Encode] Using: {os.path.basename(OUTPUT_PATH)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 6: M0 — Preprocessor (Audio + Video Metadata)  ║
# ╚══════════════════════════════════════════════════════════╝
import os, subprocess
import numpy as np
import librosa
import tensorflow_hub as hub
import pandas as pd
import cv2
from dataclasses import dataclass
from typing import List, Dict, Optional, Any
from pathlib import Path


@dataclass
class VideoMetadata:
    path:        str
    duration:    float
    fps:         float
    frame_count: int
    height:      int
    width:       int
    has_audio:   bool
    mode:        str  = "VISION_ONLY"
    audio_sr:    Optional[int] = None

class M0Preprocessor:
    """
    Audio detection + extraction + YAMNet tokenisation.
    Shared YAMNet instance avoids reloading in M1.
    """
    KEYWORD_GROUPS: Dict[str, List[str]] = {
        "emergency":      ["siren", "police", "ambulance", "fire engine",
                           "emergency vehicle", "fire alarm", "gunshot",
                           "crash", "shatter", "scream", "yell"],
        "traffic":        ["engine", "horn", "car", "vehicle", "traffic noise",
                           "truck", "bus", "motorcycle", "air brake", "air horn",
                           "tire squeal"],
        "crowd":          ["crowd", "chatter", "babble", "children playing",
                           "cheering", "applause"],
        "damage_context": ["crash", "shatter", "glass", "bang", "boom",
                           "crumpling", "thud", "explosion"],
    }
    THRESHOLDS: Dict[str, float] = {
        "emergency": 0.12, "traffic": 0.20,
        "crowd": 0.20, "damage_context": 0.18, "ambient": 0.25,
    }

    def __init__(self, audio_sr: int = 16_000,
                 window_sec: float = 0.96, hop_sec: float = 0.48,
                 top_k: int = 3):
        self.audio_sr   = audio_sr
        self.window_sec = window_sec
        self.hop_sec    = hop_sec
        self.top_k      = top_k
        self._yamnet    = None
        self._labels    = None
        self._idx_map   = {}

    def _load_yamnet(self):
        if self._yamnet is None:
            print("Loading YAMNet from TF-Hub…")
            self._yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
            class_map_path = self._yamnet.class_map_path().numpy().decode()
            self._labels   = pd.read_csv(class_map_path)["display_name"].tolist()
            for group, keywords in self.KEYWORD_GROUPS.items():
                self._idx_map[group] = [
                    i for i, lbl in enumerate(self._labels)
                    if any(kw in lbl.lower() for kw in keywords)
                ]
            print(f"YAMNet loaded | {len(self._labels)} classes")
        return self._yamnet

    def check_audio_stream(self, video_path: str) -> bool:
        result = subprocess.run(
            ["ffprobe", "-v", "quiet", "-select_streams", "a:0",
             "-show_entries", "stream=codec_name", "-of", "csv=p=0", video_path],
            capture_output=True, text=True
        )
        has = bool(result.stdout.strip())
        print(f"Audio stream present: {has}")
        return has

    def extract_audio(self, video_path: str) -> Optional[str]:
        if not self.check_audio_stream(video_path):
            return None
        stem     = Path(video_path).stem
        wav_path = os.path.join(WORKING_DIR, f"{stem}_audio.wav")
        cmd = ["ffmpeg", "-y", "-loglevel", "error",
               "-i", video_path, "-vn",
               "-acodec", "pcm_s16le", "-ar", str(self.audio_sr), "-ac", "1",
               wav_path]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0 or not os.path.exists(wav_path):
            print(f"Extraction failed: {result.stderr[-300:]}")
            return None
        size_kb = os.path.getsize(wav_path) / 1024
        if size_kb < 5:
            print(f"Audio too small ({size_kb:.1f} KB) — treating as no audio")
            os.remove(wav_path); return None
        print(f"Audio extracted → {wav_path}  ({size_kb:.0f} KB)")
        return wav_path

    def tokenise_audio(self, wav_path: str) -> List[Dict]:
        yamnet   = self._load_yamnet()
        waveform, _ = librosa.load(wav_path, sr=self.audio_sr, mono=True)
        chunk = int(self.window_sec * self.audio_sr)
        hop   = int(self.hop_sec   * self.audio_sr)
        tokens: Dict[tuple, Dict] = {}

        for i in range(0, len(waveform) - chunk + 1, hop):
            seg     = waveform[i : i + chunk]
            t_start = round(i / self.audio_sr, 3)
            t_end   = round(t_start + self.window_sec, 3)
            scores, _, _ = yamnet(seg)
            mean_s = scores.numpy().mean(axis=0)          # (521,)

            for group, indices in self._idx_map.items():
                thr = self.THRESHOLDS.get(group, 0.20)
                for idx in indices:
                    conf = float(mean_s[idx])
                    if conf >= thr:
                        key = (t_start, self._labels[idx])
                        if key not in tokens or conf > tokens[key]["confidence"]:
                            tokens[key] = {
                                "start_time": t_start, "end_time": t_end,
                                "label": self._labels[idx],
                                "confidence": round(conf, 4),
                                "type": group,
                            }

            for idx in np.argsort(mean_s)[::-1][:self.top_k]:
                conf  = float(mean_s[idx])
                label = self._labels[idx]
                key   = (t_start, label)
                if key not in tokens and conf >= self.THRESHOLDS["ambient"]:
                    tokens[key] = {
                        "start_time": t_start, "end_time": t_end,
                        "label": label,
                        "confidence": round(conf, 4),
                        "type": "ambient",
                    }

        result = sorted(tokens.values(), key=lambda t: (t["start_time"], -t["confidence"]))
        em = sum(1 for t in result if t["type"] == "emergency")
        print(f"YAMNet → {len(result)} tokens | {em} emergency")
        return result

    def get_video_metadata(self, video_path: str) -> VideoMetadata:
        cap         = cv2.VideoCapture(video_path)
        fps_v       = cap.get(cv2.CAP_PROP_FPS) or 25.0
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        has_audio = self.check_audio_stream(video_path)
        return VideoMetadata(
            path=video_path, duration=round(frame_count / fps_v, 3),
            fps=fps_v, frame_count=frame_count,
            height=height, width=width, has_audio=has_audio,
            audio_sr=self.audio_sr if has_audio else None,
        )

    def run(self, video_path: str) -> Dict[str, Any]:
        print(f"\n{'='*60}\nPreprocessing: {Path(video_path).name}\n{'='*60}")
        meta   = self.get_video_metadata(video_path)
        tokens = []
        wav    = None

        if meta.has_audio:
            wav = self.extract_audio(video_path)
            if wav:
                tokens = self.tokenise_audio(wav)
                try:    os.remove(wav)
                except: pass

        meta.mode = "AV" if tokens else "VISION_ONLY"
        print(f"Mode={meta.mode} | FPS={meta.fps:.1f} | "
              f"Duration={meta.duration:.1f}s | Tokens={len(tokens)}")
        return {"metadata": meta, "audio_tokens": tokens, "mode": meta.mode}
m0            = M0Preprocessor()
m0_out        = m0.run(OUTPUT_PATH)
video_meta    = m0_out["metadata"]
audio_tokens  = m0_out["audio_tokens"]   # List[Dict]
pipeline_mode = m0_out["mode"]

if audio_tokens:
    print("\nSample audio tokens:")
    print(pd.DataFrame(audio_tokens).sort_values("confidence", ascending=False)
          .head(10).to_string(index=False))

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 7: M1 — Perception                             ║
# ╚══════════════════════════════════════════════════════════╝
import cv2, numpy as np, pandas as pd
from ultralytics import YOLO
from pathlib import Path
from typing import List, Dict, Optional


class M1Perception:
    """YOLOv11x + ByteTrack. Audio tokens injected from M0 (no re-extraction)."""

    def __init__(self, yolo_model: str = YOLO_MODEL, max_frames: int = MAX_FRAMES):
        print("Loading YOLOv11x…")
        self.model      = YOLO(yolo_model)
        self.max_frames = max_frames
        print(f"Model ready | classes={len(self.model.names)} | max_frames={max_frames}")

    def detect(self, video_path: str, audio_tokens: List[Dict],
               fps_override: Optional[float] = None) -> Dict:
        print(f"\n{'='*60}\nDetecting: {Path(video_path).name}")

        cap = cv2.VideoCapture(video_path)
        fps = fps_override or cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release()

        results = self.model.track(
            video_path, persist=True, tracker="bytetrack.yaml",
            imgsz=640, conf=0.25, iou=0.45, verbose=False, stream=True,
        )

        detections = []
        frame_idx  = 0
        for result in results:
            if frame_idx >= self.max_frames:
                break
            if result.boxes is not None and result.boxes.id is not None:
                boxes   = result.boxes.xyxy.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy().astype(int)
                confs   = result.boxes.conf.cpu().numpy()
                ids     = result.boxes.id.cpu().numpy().astype(int)
                for i in range(len(boxes)):
                    x1, y1, x2, y2 = boxes[i]
                    detections.append({
                        "frame_idx":  frame_idx,
                        "timestamp":  round(frame_idx / fps, 3),
                        "track_id":   int(ids[i]),
                        "class_name": self.model.names[classes[i]],
                        "confidence": round(float(confs[i]), 4),
                        "bbox":       boxes[i],
                        "x1": float(x1), "y1": float(y1),
                        "x2": float(x2), "y2": float(y2),
                        "bbox_x": float((x1 + x2) / 2),
                        "bbox_y": float((y1 + y2) / 2),
                    })
            frame_idx += 1

        df = pd.DataFrame(detections) if detections else pd.DataFrame()

        vehicle_cls = {"car","truck","bus","motorcycle","bicycle"}
        scene_stats = {
            "total_detections": len(df),
            "unique_tracks":    df["track_id"].nunique() if not df.empty else 0,
            "vehicle_tracks":   df[df["class_name"].isin(vehicle_cls)]["track_id"].nunique() if not df.empty else 0,
            "person_tracks":    df[df["class_name"]=="person"]["track_id"].nunique() if not df.empty else 0,
            "emergency_audio":  any(t["type"]=="emergency" for t in audio_tokens),
            "mode":             pipeline_mode,
        }
        print(f"Done → {scene_stats['unique_tracks']} tracks | "
              f"{len(audio_tokens)} audio tokens | mode={pipeline_mode}")
        return {"detections_df": df, "audio_tokens": audio_tokens,
                "scene_stats": scene_stats, "fps": fps}
m1            = M1Perception(max_frames=MAX_FRAMES)
m1_out        = m1.detect(OUTPUT_PATH, audio_tokens, fps_override=video_meta.fps)
detections_df = m1_out["detections_df"]
fps           = m1_out["fps"]
scene_stats   = m1_out["scene_stats"]
# ── Post-tracking: deduplicate ID swaps and re-identify occluded tracks ──
detections_df            = deduplicate_and_reidentify(detections_df, fps=fps)
scene_stats["unique_tracks"] = int(detections_df["track_id"].nunique())

print("\n[Scene Stats]")
for k, v in scene_stats.items(): print(f"  {k}: {v}")
print("\n[Detections Sample]")
print(detections_df.head(8).to_string(index=False))

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 8: M2 — Tracking                               ║
# ╚══════════════════════════════════════════════════════════╝
import numpy as np, pandas as pd
from typing import Dict


class M2Tracking:
    """Enriches detections_df with centroid, velocity, acceleration, direction."""

    def __init__(self):
        print("Initialised → Motion & Trajectory Features")

    def track(self, detections_df: pd.DataFrame, fps: float) -> pd.DataFrame:
        if detections_df.empty:
            print("⚠ Empty detections_df"); return pd.DataFrame()

        df = detections_df.sort_values(["track_id", "timestamp"]).copy()
        track_state: Dict[int, tuple] = {}
        rows = []

        for _, row in df.iterrows():
            tid = int(row["track_id"])
            ts  = float(row["timestamp"])
            cx  = float(row.get("bbox_x", (row["x1"] + row["x2"]) / 2))
            cy  = float(row.get("bbox_y", (row["y1"] + row["y2"]) / 2))

            if tid in track_state:
                prev_ts, prev_cx, prev_cy, prev_vx, prev_vy = track_state[tid]
                dt  = max(ts - prev_ts, 1e-6)
                vx  = (cx - prev_cx) / dt
                vy  = (cy - prev_cy) / dt
                spd = float(np.hypot(vx, vy))
                acc = (spd - float(np.hypot(prev_vx, prev_vy))) / dt
                direction_deg = float(np.degrees(np.arctan2(vy, vx))) % 360
            else:
                vx = vy = spd = acc = direction_deg = 0.0

            track_state[tid] = (ts, cx, cy, vx, vy)
            rows.append({
                **row.to_dict(),
                "centroid_x":    round(cx, 2),
                "centroid_y":    round(cy, 2),
                "vx":            round(vx, 3),
                "vy":            round(vy, 3),
                "velocity":      round(spd, 3),
                "acceleration":  round(acc, 3),
                "direction_deg": round(direction_deg, 1),
            })

        tracks_df = pd.DataFrame(rows)
        print(f"Done → {tracks_df['track_id'].nunique()} tracks | "
              f"max velocity={tracks_df['velocity'].max():.1f} px/s")
        return tracks_df
m2        = M2Tracking()
tracks_df = m2.track(detections_df, fps=fps)

print("\n[Tracks Sample]")
print(tracks_df[["frame_idx","timestamp","track_id","class_name",
                  "velocity","acceleration","direction_deg"]].head(10).to_string(index=False))

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 9: BBox Visualization — Static Plots           ║
# ╚══════════════════════════════════════════════════════════╝
import cv2, os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Video, display
from typing import Tuple


def plot_bbox_grid(video_path: str, tracks_df: pd.DataFrame,
                   n_frames: int = 8, figsize=(20, 10)):
    """Sample n_frames evenly, draw bounding boxes, display as a grid."""
    cap        = cv2.VideoCapture(video_path)
    total      = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_v      = cap.get(cv2.CAP_PROP_FPS) or 25.0
    targets    = np.linspace(0, min(total - 1, MAX_FRAMES - 1), n_frames, dtype=int)

    fig, axes  = plt.subplots(2, n_frames // 2, figsize=figsize)
    fig.patch.set_facecolor("#111111")

    for ax_idx, frame_target in enumerate(targets):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_target))
        ret, frame = cap.read()
        if not ret: continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ts        = round(frame_target / fps_v, 2)
        frame_tracks = tracks_df[tracks_df["frame_idx"] == frame_target]
        if frame_tracks.empty:
            nearest_fidx = (tracks_df["frame_idx"] - frame_target).abs().idxmin()
            frame_tracks = tracks_df[tracks_df["frame_idx"] == tracks_df.loc[nearest_fidx, "frame_idx"]]

        ax = axes.flat[ax_idx]
        ax.imshow(frame_rgb)
        ax.set_title(f"t={ts:.2f}s  |  frame {frame_target}", color="white", fontsize=8, pad=3)
        ax.axis("off")

        color_norm = (0/255, 200/255, 0/255)
        for _, t in frame_tracks.iterrows():
            tid  = int(t["track_id"])
            x1, y1, x2, y2 = get_bbox(t)
            cls  = str(t["class_name"])
            conf = float(t.get("confidence", 0.0))
            spd  = float(t.get("velocity", 0.0))
            rect = mpatches.FancyBboxPatch(
                (x1, y1), x2-x1, y2-y1,
                boxstyle="square,pad=0", linewidth=2,
                edgecolor=color_norm, facecolor="none"
            )
            ax.add_patch(rect)
            label = f"#{tid} {cls} {conf:.2f}"
            if spd > 1: label += f" v={spd:.0f}"
            ax.text(x1, max(y1-4, 0), label, color="white", fontsize=6.5,
                    fontweight="bold",
                    bbox=dict(facecolor=color_norm, alpha=0.75, edgecolor="none", pad=1.5))
            vx = float(t.get("vx", 0.0)); vy = float(t.get("vy", 0.0))
            if spd > 3:
                cx = (x1+x2)/2; cy = (y1+y2)/2; sc = min(40, spd*0.3)
                ax.annotate("",
                    xy=(cx+vx/spd*sc, cy+vy/spd*sc), xytext=(cx, cy),
                    arrowprops=dict(arrowstyle="->", color=color_norm, lw=1.8, mutation_scale=12))

    cap.release()
    unique_classes = tracks_df[["track_id","class_name"]].drop_duplicates()
    legend_patches = [
        mpatches.Patch(color=(0/255, 200/255, 0/255),
                       label=f"#{int(r['track_id'])} {r['class_name']}")
        for _, r in unique_classes.iterrows()
    ]
    fig.legend(handles=legend_patches[:16], loc="lower center",
               ncol=min(8, len(legend_patches)), facecolor="#222",
               edgecolor="#555", labelcolor="white", fontsize=7.5,
               bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(
        f"Bounding Box Grid — {os.path.basename(video_path)}  "
        f"| {tracks_df['track_id'].nunique()} tracks",
        color="white", fontsize=12, fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0.06, 1, 0.97])
    out = os.path.join(WORKING_DIR, "bbox_frame_grid.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#111111")
    plt.show()
    print(f"[BBoxVis] Frame grid saved → {out}")


def plot_velocity_heatmap(tracks_df: pd.DataFrame, figsize=(18, 6)):
    """Heatmap of velocity (px/s) per track over time."""
    if "velocity" not in tracks_df.columns:
        print("[BBoxVis] No velocity column — skipping heatmap"); return

    pivot = tracks_df.pivot_table(
        index="track_id", columns="frame_idx",
        values="velocity", aggfunc="mean"
    ).fillna(0)

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("#111111"); ax.set_facecolor("#111111")
    im = ax.imshow(pivot.values, aspect="auto", cmap="plasma",
                   interpolation="nearest",
                   vmin=0, vmax=float(tracks_df["velocity"].quantile(0.95)))

    ax.set_yticks(range(len(pivot.index)))
    track_labels = [
        f"#{tid} {tracks_df[tracks_df['track_id']==tid]['class_name'].iloc[0]}"
        for tid in pivot.index
    ]
    ax.set_yticklabels(track_labels, color="white", fontsize=8)
    n_cols = pivot.shape[1]; step = max(1, n_cols//20)
    ax.set_xticks(range(0, n_cols, step))
    ax.set_xticklabels(
        [str(pivot.columns[i]) for i in range(0, n_cols, step)],
        color="white", fontsize=7, rotation=45
    )
    ax.set_xlabel("Frame Index", color="white")
    ax.set_ylabel("Track", color="white")
    ax.set_title("Track Velocity Heatmap (px/s) — brighter = faster",
                 color="white", fontsize=12, fontweight="bold")
    ax.tick_params(colors="white")
    cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label("Velocity (px/s)", color="white")
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
    plt.tight_layout()
    out = os.path.join(WORKING_DIR, "velocity_heatmap.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#111111")
    plt.show()
    print(f"[BBoxVis] Velocity heatmap saved → {out}")


# ── Execution ──────────────────────────────────────────────
print("\n" + "="*60)
print("[BBoxVis] Static Visualizations")
print("="*60)
print(f"  Tracks  : {tracks_df['track_id'].nunique()} unique")
print(f"  Classes : {tracks_df['class_name'].unique().tolist()}")

print("\n[BBoxVis] Generating frame grid...")
plot_bbox_grid(OUTPUT_PATH, tracks_df, n_frames=8)

print("\n[BBoxVis] Generating velocity heatmap...")
plot_velocity_heatmap(tracks_df)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 10: Bayesian AV Fusion                         ║
# ╚══════════════════════════════════════════════════════════╝
import numpy as np, pandas as pd
from scipy.special import logsumexp
from typing import Dict, List
class BayesianAVFusion:
    """
    Temporal Bayesian fusion of visual (YOLO) + audio (YAMNet) evidence.
    Class set:  car(0) truck(1) bus(2) ambulance(3) police(4)
                fire_truck(5) motorcycle(6) bicycle(7) person(8) other(9)
    """
    CLASS_NAMES    = ["car","truck","bus","ambulance","police","fire_truck",
                      "motorcycle","bicycle","person","other"]
    EMERGENCY_IDX  = {3, 4, 5}
    YOLO_TO_IDX: Dict[str, int] = {
        "car":0,"truck":1,"bus":2,"ambulance":3,"police car":3,"police":4,
        "fire engine":5,"fire truck":5,"motorcycle":6,"bicycle":7,"person":8,
    }

    def __init__(self):
        K = len(self.CLASS_NAMES); self.K = K
        prior = np.array([0.30,0.12,0.06,0.03,0.03,0.02,0.06,0.04,0.25,0.09])
        prior /= prior.sum()
        self.log_prior = np.log(prior + 1e-15)

        T = np.eye(K) * 0.96 + 0.004
        T /= T.sum(axis=1, keepdims=True)
        self.log_T = np.log(T + 1e-15)

        C = np.eye(K) * 0.85
        C[3,1]=0.10; C[5,1]=0.08; C[4,0]=0.07
        C /= C.sum(axis=1, keepdims=True)
        self.visual_C = C

        self.p_siren_given_class = np.array(
            [0.005,0.005,0.003,0.92,0.88,0.82,0.003,0.001,0.002,0.002])
        self._beliefs: Dict[int, np.ndarray] = {}
        print("Bayesian AV Fusion initialised")

    @staticmethod
    def _siren_prob(audio_tokens: List[Dict], ts: float, window: float = 0.5) -> float:
        scores = [t["confidence"] for t in audio_tokens
                  if t.get("type") == "emergency"
                  and abs((t["start_time"]+t["end_time"])/2 - ts) <= window]
        return float(max(scores)) if scores else 0.0

    def _update(self, tid: int, pred_idx: int, raw_conf: float, siren_p: float) -> np.ndarray:
        prev = self._beliefs.get(tid, self.log_prior.copy())
        log_pred = logsumexp(self.log_T.T + prev, axis=1)

        lv = np.clip(raw_conf, 0.1, 0.95)
        vis_ll  = lv * self.visual_C[:, pred_idx] + (1-lv) / self.K
        aud_ll  = siren_p * self.p_siren_given_class + (1-siren_p)*(1-self.p_siren_given_class)

        log_post = log_pred + np.log(vis_ll + 1e-15) + np.log(aud_ll + 1e-15)
        log_post -= logsumexp(log_post)
        self._beliefs[tid] = log_post
        return np.exp(log_post)

    def fuse(self, tracks_df: pd.DataFrame, audio_tokens: List[Dict]) -> pd.DataFrame:
        if tracks_df.empty:
            print("⚠ Empty tracks_df"); return pd.DataFrame()

        self._beliefs.clear()
        out_frames = []

        for fidx in sorted(tracks_df["frame_idx"].unique()):
            frame  = tracks_df[tracks_df["frame_idx"] == fidx]
            rows   = []
            for _, det in frame.iterrows():
                tid      = int(det["track_id"])
                raw_name = str(det["class_name"]).lower()
                pred_idx = self.YOLO_TO_IDX.get(raw_name, self.K-1)
                raw_conf = float(det.get("confidence", 0.5))
                ts       = float(det["timestamp"])

                sp = self._siren_prob(audio_tokens, ts)
                posterior = self._update(tid, pred_idx, raw_conf, sp)
                best_idx  = int(np.argmax(posterior))

                row = det.to_dict()
                row.update({
                    "class_name":   self.CLASS_NAMES[best_idx],
                    "confidence":   round(float(posterior[best_idx]), 4),
                    "yolo_class":   raw_name,
                    "is_emergency": best_idx in self.EMERGENCY_IDX,
                    "siren_prob":   round(sp, 4),
                })
                rows.append(row)
            out_frames.append(pd.DataFrame(rows))

        master_df = pd.concat(out_frames, ignore_index=True)
        n_em = master_df["is_emergency"].sum()
        print(f" Fusion done → {len(master_df)} rows | {n_em} emergency detections")
        return master_df
av_fusion    = BayesianAVFusion()
master_df = av_fusion.fuse(tracks_df, audio_tokens)

print("\n[Master DF (unique tracks)]")
print(master_df[["track_id","yolo_class","class_name","confidence",
                  "is_emergency","siren_prob"]]
      .drop_duplicates("track_id").head(15).to_string(index=False))

em_tracks = master_df[master_df["is_emergency"]]
if not em_tracks.empty:
    print(f"\n ⚠ Emergency tracks: {em_tracks['track_id'].unique().tolist()}")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 11: M3 - Relational Inference                  ║
# ╚══════════════════════════════════════════════════════════╝
import numpy as np, pandas as pd, math
from typing import List, Dict


class M3RelationInference:
    """
    Rule-based spatial + audio-visual triplet generator.
    Thresholds are auto-calibrated per scene type detected from master_df.

    Scene types:
      'vehicle_road'   — cars/trucks dominant (dashcam, traffic footage)
      'mixed'          — vehicles + pedestrians (intersection, parking lot)
      'pedestrian'     — people only (metro, mall, stadium, sidewalk)

    Predicate set: Colliding | Approaching | Overtaking | Following |
                   Near | Far | HasStatus_Emergency | EmergencyYielding |
                   EmergencySound | TrafficSound | CrowdSound |
                   DamageContextSound | AmbientSound
    """

    AUDIO_PRED_MAP = {
        "emergency":      "EmergencySound",
        "traffic":        "TrafficSound",
        "crowd":          "CrowdSound",
        "damage_context": "DamageContextSound",
        "ambient":        "AmbientSound",
    }

    # ── Per-scene threshold profiles ─────────────────────────────────────────
    SCENE_PROFILES = {
        "vehicle_road": {
            "COLLISION_DIST":  0.70,   # vehicles are large, overlap = crash
            "APPROACH_DIST":   4.50,
            "FOLLOW_COS":      0.82,
            "NEAR_DIST":       3.00,
            "MIN_SPEED":       0.80,
        },
        "mixed": {
            "COLLISION_DIST":  0.40,   # tighter — not every close pass is a crash
            "APPROACH_DIST":   3.00,
            "FOLLOW_COS":      0.82,
            "NEAR_DIST":       2.00,
            "MIN_SPEED":       1.50,
        },
        "pedestrian": {
            "COLLISION_DIST":  0.15,   # only genuine bbox overlap counts
            "APPROACH_DIST":   2.00,   # crowded spaces, approach is normal
            "FOLLOW_COS":      0.85,
            "NEAR_DIST":       1.50,   # everyone is near in a station/mall
            "MIN_SPEED":       3.00,   # slow drift is not meaningful motion
        },
    }

    def __init__(self, frame_width: int = 640, frame_height: int = 640):
        self.fw = frame_width
        self.fh = frame_height
        self.H  = np.array([[10, 0, 0], [0, 10, 0], [0, 0, 1]], dtype=float)

        # Defaults — will be overridden by _detect_scene_type()
        self.COLLISION_DIST = 0.70
        self.APPROACH_DIST  = 4.50
        self.FOLLOW_COS     = 0.82
        self.NEAR_DIST      = 3.00
        self.MIN_SPEED      = 0.80
        self.scene_type     = "vehicle_road"

        print("Initialised → Spatial + Audio-Visual Relation Inference")

    # ── Scene type detection ──────────────────────────────────────────────────

    VEHICLE_CLASSES = frozenset({
        "car", "truck", "bus", "ambulance", "police",
        "fire_truck", "motorcycle", "bicycle"
    })

    def _detect_scene_type(self, master_df: pd.DataFrame) -> str:
        """
        Infers scene type from the class distribution in master_df.
        Uses unique track counts so one car with 500 detections
        doesn't dominate the ratio.

        Returns: 'vehicle_road' | 'mixed' | 'pedestrian'
        """
        if master_df.empty:
            return "vehicle_road"  # safe default

        unique = master_df.drop_duplicates("track_id")[["track_id", "class_name"]]
        total  = len(unique)
        if total == 0:
            return "vehicle_road"

        n_vehicle = unique["class_name"].isin(self.VEHICLE_CLASSES).sum()
        n_person  = (unique["class_name"] == "person").sum()
        n_other   = total - n_vehicle - n_person

        veh_ratio    = n_vehicle / total
        person_ratio = (n_person + n_other) / total  # 'other' = unclassified people

        if veh_ratio >= 0.60:
            scene = "vehicle_road"
        elif veh_ratio >= 0.20:
            scene = "mixed"
        else:
            scene = "pedestrian"

        print(f"  [M3] Scene detection → {scene} "
              f"(vehicles={n_vehicle}, persons={n_person}, "
              f"other={n_other}, total_tracks={total})")
        print(f"       veh_ratio={veh_ratio:.2f}  person_ratio={person_ratio:.2f}")
        return scene

    def _apply_profile(self, scene_type: str):
        """Load threshold profile for detected scene type."""
        profile = self.SCENE_PROFILES[scene_type]
        self.COLLISION_DIST = profile["COLLISION_DIST"]
        self.APPROACH_DIST  = profile["APPROACH_DIST"]
        self.FOLLOW_COS     = profile["FOLLOW_COS"]
        self.NEAR_DIST      = profile["NEAR_DIST"]
        self.MIN_SPEED      = profile["MIN_SPEED"]
        print(f"  [M3] Thresholds applied for '{scene_type}': "
              f"COLLISION={self.COLLISION_DIST} | "
              f"APPROACH={self.APPROACH_DIST} | "
              f"NEAR={self.NEAR_DIST} | "
              f"MIN_SPEED={self.MIN_SPEED}")

    # ── Projection ────────────────────────────────────────────────────────────

    def _project(self, nx_: float, ny_: float) -> np.ndarray:
        p = self.H @ np.array([nx_, ny_, 1.0])
        return p[:2]

    # ── Main inference ────────────────────────────────────────────────────────

    def infer_triplets(self, master_df: pd.DataFrame,
                       audio_tokens: List[Dict],
                       fps: float = 25.0) -> pd.DataFrame:

        # Auto-detect scene and load matching thresholds
        self.scene_type = self._detect_scene_type(master_df)
        self._apply_profile(self.scene_type)

        relations = []

        for ts, frame_data in master_df.groupby("timestamp"):
            tracks = frame_data.to_dict("records")
            if len(tracks) < 2:
                continue

            for t in tracks:
                cx = float(t.get("bbox_x", (t["x1"] + t["x2"]) / 2)) / self.fw
                cy = float(t.get("bbox_y", (t["y1"] + t["y2"]) / 2)) / self.fh
                t["_pos"] = self._project(cx, cy)
                t["_vel"] = np.array([t.get("vx", 0.0), t.get("vy", 0.0)])
                t["_spd"] = float(np.linalg.norm(t["_vel"]))

            for i, a in enumerate(tracks):
                pos_a = a["_pos"]
                vel_a = a["_vel"]
                spd_a = a["_spd"]
                subj  = f"{a['class_name']}_{a['track_id']}"

                if a.get("is_emergency", False):
                    relations.append({
                        "subject":    subj,
                        "predicate":  "HasStatus_Emergency",
                        "object":     "EmergencyResponding",
                        "confidence": round(float(a.get("confidence", 0.9)), 3),
                        "timestamp":  ts,
                    })

                for j, b in enumerate(tracks):
                    if i == j:
                        continue
                    pos_b = b["_pos"]
                    vel_b = b["_vel"]
                    spd_b = b["_spd"]
                    obj   = f"{b['class_name']}_{b['track_id']}"
                    dist  = float(math.dist(pos_a, pos_b))
                    conf  = float(a.get("confidence", 0.8))
                    pred  = None

                    # ── Collision: in pedestrian scenes, ALSO require
                    #    both entities to be moving (not just standing close)
                    if (dist < self.COLLISION_DIST
                            and spd_a > self.MIN_SPEED
                            and spd_b > self.MIN_SPEED):
                        pred = "Colliding"
                        conf = min(0.98, conf + 0.1)

                    elif spd_a > self.MIN_SPEED and dist < self.APPROACH_DIST:
                        if np.dot(vel_a, pos_b - pos_a) > 0:
                            pred = "Approaching"

                    elif spd_a > self.MIN_SPEED and spd_b > self.MIN_SPEED:
                        cos_sim = np.dot(vel_a, vel_b) / (spd_a * spd_b + 1e-9)
                        if cos_sim > self.FOLLOW_COS:
                            pred = ("Overtaking"
                                    if spd_a > spd_b * 1.3 and dist < self.NEAR_DIST
                                    else "Following")

                    elif dist < self.NEAR_DIST:
                        pred = "Near"
                    else:
                        pred = "Far"

                    if pred:
                        relations.append({
                            "subject":    subj,
                            "predicate":  pred,
                            "object":     obj,
                            "confidence": round(conf, 3),
                            "timestamp":  ts,
                        })

                # EmergencyYielding — only in vehicle/mixed scenes
                if (a.get("is_emergency", False)
                        and self.scene_type in ("vehicle_road", "mixed")):
                    for j, b in enumerate(tracks):
                        if i == j or b["class_name"] != "person":
                            continue
                        if float(math.dist(pos_a, b["_pos"])) < self.NEAR_DIST:
                            relations.append({
                                "subject":    f"{b['class_name']}_{b['track_id']}",
                                "predicate":  "EmergencyYielding",
                                "object":     subj,
                                "confidence": 0.88,
                                "timestamp":  ts,
                            })

        # Audio tokens
        for token in audio_tokens:
            ts_a  = (token["start_time"] + token["end_time"]) / 2
            pred  = self.AUDIO_PRED_MAP.get(token.get("type", "ambient"), "AmbientSound")
            near  = master_df[abs(master_df["timestamp"] - ts_a) < 0.5]
            if near.empty:
                continue
            if token.get("type") == "emergency" and "is_emergency" in near.columns:
                em = near[near["is_emergency"].astype(bool)]
                if not em.empty:
                    near = em
            t = near.iloc[0]
            relations.append({
                "subject":    f"{t['class_name']}_{t['track_id']}",
                "predicate":  pred,
                "object":     token["label"],
                "confidence": round(float(token["confidence"]), 3),
                "timestamp":  ts_a,
            })

        if not relations:
            print("⚠ No relations generated")
            return pd.DataFrame(
                columns=["subject", "predicate", "object", "confidence", "timestamp"])

        relations_df = pd.DataFrame(relations)
        print(f"  Done → {len(relations_df)} relations")
        print(f"  Scene type used: {self.scene_type}")
        print(f"  Predicates: {relations_df['predicate'].value_counts().to_dict()}")
        return relations_df


m3           = M3RelationInference(frame_width=video_meta.width,
                                   frame_height=video_meta.height)
relations_df = m3.infer_triplets(master_df, audio_tokens, fps=fps)

print("\n[Relations Sample]")
print(relations_df.head(15).to_string(index=False))

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 12: M4 - Dynamic Knowledge Graph               ║
# ╚══════════════════════════════════════════════════════════╝
import networkx as nx, numpy as np, pandas as pd
from collections import defaultdict
from typing import List, Dict
class M4DynamicKnowledgeGraph:
    CRITICAL_PREDS = frozenset({
        "Colliding","Approaching","EmergencyYielding",
        "HasStatus_Emergency","EmergencyResponse","Overtaking",
    })

    def __init__(self, smoothing_alpha: float = 0.65,
                 prune_age: float = 5.0, min_confidence: float = 0.10):
        self.alpha   = smoothing_alpha
        self.prune_age = prune_age
        self.min_conf  = min_confidence
        self.G       = nx.MultiDiGraph()
        self._tindex: Dict[float, List[str]] = defaultdict(list)
        print(f" DKG initialised | α={smoothing_alpha} | prune_age={prune_age}s")
    def _obj_node(self, obj_id: str, class_name: str, ts: float,
                  is_em: bool = False) -> str:
        nid = f"{obj_id}@{ts:.3f}"
        if not self.G.has_node(nid):
            self.G.add_node(nid, type="object", label=obj_id,
                            class_name=class_name, timestamp=ts, is_emergency=is_em)
            self._tindex[ts].append(nid)
        return nid

    def _aud_node(self, label: str, ts: float, conf: float) -> str:
        nid = f"audio_{label.replace(' ','_')}@{ts:.3f}"
        if not self.G.has_node(nid):
            self.G.add_node(nid, type="audio_event", label=label,
                            confidence=conf, timestamp=ts)
        return nid

    def _scene_node(self, ts: float) -> str:
        nid = f"scene@{ts:.3f}"
        if not self.G.has_node(nid): self.G.add_node(nid, type="scene", timestamp=ts)
        return nid
    def _upsert(self, src: str, dst: str, pred: str, conf: float, ts: float):
        if conf < self.min_conf: return
        if self.G.has_edge(src, dst):
            for key, data in self.G[src][dst].items():
                if data.get("predicate") == pred and (ts - data.get("last_seen",0)) <= 2.0:
                    alpha = 0.85 if pred in self.CRITICAL_PREDS else self.alpha
                    self.G[src][dst][key]["confidence"] = round(
                        alpha*conf + (1-alpha)*data["confidence"], 4)
                    self.G[src][dst][key]["last_seen"] = ts
                    return
        self.G.add_edge(src, dst, predicate=pred,
                        confidence=round(conf,4), first_seen=ts, last_seen=ts)

    def prune(self, current_time: float):
        stale = [(u,v,k) for u,v,k,d in self.G.edges(keys=True,data=True)
                 if (current_time - d.get("last_seen",0)) > self.prune_age]
        self.G.remove_edges_from(stale)
        self.G.remove_nodes_from(list(nx.isolates(self.G)))

    def build_from_relations(self, master_df: pd.DataFrame,
                              relations_df: pd.DataFrame,
                              audio_tokens: List[Dict],
                              fps: float = 25.0) -> nx.MultiDiGraph:
        print("\n Building Dynamic Knowledge Graph…")
        self.G.clear(); self._tindex.clear()

        # Object + scene nodes
        for _, row in master_df.iterrows():
            ts    = round(float(row["timestamp"]), 3)
            obj_id= f"{row['class_name']}_{row['track_id']}"
            is_em = bool(row.get("is_emergency", False))
            node  = self._obj_node(obj_id, row["class_name"], ts, is_em)
            scene = self._scene_node(ts)
            self._upsert(node, scene, "AppearsIn", 1.0, ts)

        # Relations from M3
        for _, row in relations_df.iterrows():
            ts   = round(float(row["timestamp"]), 3)
            subj = f"{row['subject']}@{ts:.3f}"
            raw_obj = str(row["object"])

            # Determine whether the object is an audio label or a tracked entity
            last_part = raw_obj.split("_")[-1]
            is_tracked_entity = last_part.isdigit()

            if not is_tracked_entity:
                obj = self._aud_node(raw_obj, ts, float(row["confidence"]))
            else:
                obj = f"{raw_obj}@{ts:.3f}"
                if not self.G.has_node(obj):
                    parts = raw_obj.rsplit("_", 1)
                    self._obj_node(raw_obj, parts[0] if len(parts) > 1 else raw_obj, ts)

            if self.G.has_node(subj):
                self._upsert(subj, obj, row["predicate"],float(row["confidence"]), ts)

        # Audio event nodes linked to nearest objects
        for token in audio_tokens:
            ts_a = round((token["start_time"]+token["end_time"])/2, 3)
            aud  = self._aud_node(token["label"], ts_a, token["confidence"])
            nearby = master_df[abs(master_df["timestamp"] - ts_a) < 0.5]
            for _, t in nearby.iterrows():
                obj_nid = f"{t['class_name']}_{t['track_id']}@{round(t['timestamp'],3):.3f}"
                if self.G.has_node(obj_nid):
                    self._upsert(aud, obj_nid, "Affects",
                                 float(token["confidence"]), ts_a)

        print(f" Built → {self.G.number_of_nodes()} nodes | {self.G.number_of_edges()} edges")
        node_types = {}
        for _, d in self.G.nodes(data=True):
            node_types[d.get("type","?")] = node_types.get(d.get("type","?"),0)+1
        print(f" Node types: {node_types}")
        return self.G

    def get_active_triplets(self) -> List[Dict]:
        return [{"subject":u,"predicate":d["predicate"],"object":v,
                 "confidence":d["confidence"],"last_seen":d.get("last_seen",0)}
                for u,v,d in self.G.edges(data=True)]

    def snapshot_at(self, ts: float, window: float = 0.1) -> nx.MultiDiGraph:
        nodes = [n for n,d in self.G.nodes(data=True)
                 if abs(d.get("timestamp",-999)-ts) <= window]
        return self.G.subgraph(nodes).copy()


scene_type_detected = m3.scene_type
m4 = M4DynamicKnowledgeGraph(smoothing_alpha=0.65, prune_age=5.0)
KG = m4.build_from_relations(master_df, relations_df, audio_tokens, fps=fps)

active_triplets = m4.get_active_triplets()
print(f"\n Active triplets: {len(active_triplets)}")
if active_triplets:
    print(pd.DataFrame(active_triplets)
          .sort_values("confidence", ascending=False).head(12).to_string(index=False))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell 13: M5 — Event Engine                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
from typing import List, Dict, Set

VEHICLE_CLASSES = frozenset({"car", "truck", "bus", "ambulance", "police",
                              "fire_truck", "motorcycle", "bicycle"})

def _is_vehicle(name: str) -> bool:
    return any(vc in name.lower() for vc in VEHICLE_CLASSES)

def _tid_from_name(name: str) -> int:
    try: return int(name.split("_")[-1])
    except: return -1
# ──────────────────────────────────────────────────────────────────────────────
# VALIDATION LAYER
# Every detector runs its output through this before returning.
# ──────────────────────────────────────────────────────────────────────────────

class EventValidator:
    """
    Hard gates applied to every event before it leaves a detector.

    CONFIDENCE_FLOORS: minimum confidence per event type regardless of
    what the detector computed. Critical events need high evidence.

    REQUIRED_SIGNALS: for events that claim multi-modal evidence,
    these keys must be present and truthy in the event dict.
    """

    CONFIDENCE_FLOORS: Dict[str, float] = {
        # Category 1 — Emergency
        "AccidentCollisionDetection":  0.80,   
        "EmergencyVehicleTracking":    0.70,
        "StalledBrokenDownVehicle":    0.75,
        "MedicalEmergency":            0.72,   
        # Category 2 — Traffic
        "WrongSideDriving":            0.72,
        "Speeding":                    0.75,
        "IllegalParking":              0.68,
        "LaneDisciplineViolation":     0.65,
        "TrafficSignalSignBreach":     0.70,
        "SafetyGearNonCompliance":     0.60,
        # Category 3 — Security
        "VehicleTheft":                0.72,
        "RobberySnatching":            0.75,   
        "AbandonedObject":             0.65,
        # Category 4 — Operational
        "TrafficCongestion":           0.60,
        "CrowdMonitoring":             0.58,
        "InfrastructureDamage":        0.65,
    }

    def validate(self, events: List[Dict]) -> List[Dict]:
        """Drop events below their type-specific confidence floor."""
        passed = []
        for ev in events:
            floor = self.CONFIDENCE_FLOORS.get(ev["event_type"], 0.65)
            if ev["confidence"] >= floor:
                passed.append(ev)
            # Uncomment to debug which events are being filtered:
            # else:
            #     print(f"  [VALIDATOR] Dropped {ev['event_type']} "
            #           f"conf={ev['confidence']:.3f} < floor={floor:.3f}")
        return passed


# ──────────────────────────────────────────────────────────────────────────────
# CROSS-DETECTOR MUTEX  
# Prevents the same track_id from firing multiple overlapping event types
# that describe the same physical state (e.g. stopped car can't be both
# StalledBrokenDownVehicle AND AbandonedObject AND VehicleTheft).
# ──────────────────────────────────────────────────────────────────────────────

# Groups: within each group, only the highest-confidence event per track fires.
MUTEX_GROUPS = [
    {"StalledBrokenDownVehicle", "AbandonedObject", "VehicleTheft", "IllegalParking"},
    {"MedicalEmergency", "RobberySnatching"},      # stationary person can't be both
    {"LaneDisciplineViolation", "WrongSideDriving"},
]


def apply_mutex(events: List[Dict]) -> List[Dict]:
    """
    For each mutex group, extract the track_id from the object field
    and keep only the single highest-confidence event per track.
    """
    def _tid(obj_str: str) -> str:
        # Normalise: 'car_3 vs truck_7' → 'car_3|truck_7', 'person_6' → 'person_6'
        return obj_str.split("(")[0].strip()

    surviving = list(events)
    for group in MUTEX_GROUPS:
        group_events = [(i, e) for i, e in enumerate(surviving)
                        if e["event_type"] in group]
        # Group by normalised object id
        by_obj: Dict[str, List] = {}
        for idx, ev in group_events:
            key = _tid(ev["object"])
            by_obj.setdefault(key, []).append((idx, ev))
        # Within each object keep only the top-confidence event
        to_remove: Set[int] = set()
        for obj_key, candidates in by_obj.items():
            if len(candidates) > 1:
                best_idx = max(candidates, key=lambda x: x[1]["confidence"])[0]
                for idx, _ in candidates:
                    if idx != best_idx:
                        to_remove.add(idx)
        surviving = [e for i, e in enumerate(surviving) if i not in to_remove]
    return surviving


# ──────────────────────────────────────────────────────────────────────────────
# MAIN ENGINE
# ──────────────────────────────────────────────────────────────────────────────

class M5EventEngine:
    """
    Anti-hallucination event engine.
    Each detector documents its AND-gate logic and exclusion rules.
    """

    # ── Thresholds ──────────────────────────────────
    SPEED_STALL              = 2.0
    SPEED_FAST               = 200.0
    NEAR_THRESH              = 120
    BRAKE_DECEL_THRESH       = -40.0
    BRAKE_MIN_SPEED          = 40.0
    WRONG_WAY_COS            = -0.75
    WRONG_WAY_MIN_SPD        = 10.0
    CROWD_PERSON_THRESH      = 5
    CROWD_AUDIO_THRESH       = 0.25
    REDLIGHT_STOP_FRAC       = 0.85
    REDLIGHT_MIN_SPD         = 20.0
    PARKING_MIN_FRAMES       = 10
    STATIONARY_PERSON_FRAMES = 8
    ABANDONED_OBJ_FRAMES     = 15
    PERSON_OBJ_DIST_PX       = 100
    INFRA_AUDIO_KEYWORDS     = {"crash", "shatter", "glass", "bang", "boom",
                                 "crumpling", "thud", "explosion", "scraping"}
    THEFT_SPEED_THRESH       = 80.0
    THEFT_STATIONARY_FRAMES  = 5
    SNATCHING_ACCEL_THRESH   = 30.0
    DISTRACTED_LATERAL_LOW   = 3.0
    DISTRACTED_LATERAL_HIGH  = 15.0
    DISTRACTED_DURATION_FRAMES = 5
    MEDICAL_FALL_SPEED       = 5.0
    MEDICAL_AUDIO_KW         = {"scream", "yell", "cry", "groan", "fall"}
    INTRUSION_ZONE_Y_FRAC    = 0.20
    HELMET_AUDIO_KW          = {"helmet", "motorcycle"}

    # ── Minimum sustained-frame requirements ──────────────────────────────
    MIN_FRAMES_WRONG_WAY     = 8     # must be going wrong way for 8+ frames
    MIN_FRAMES_SPEEDING      = 10     # sustained speed, not a single frame spike
    MIN_FRAMES_DISTRACTED    = 8     # wobble must persist (raised from 5)
    MIN_FRAMES_THEFT_BURST   = 3     # burst must last 3+ frames (not one blip)
    MIN_FRAMES_SNATCH        = 2     # accel spike must persist
    MIN_FRAMES_INTRUSION     = 4     # person must be in zone for 4+ frames

    def __init__(self):
        self.validator = EventValidator()
        print("M5EventEngine (anti-hallucination) initialised")

    def _validate(self, events: List[Dict]) -> List[Dict]:
        return self.validator.validate(events)

    # ══════════════════════════════════════════════════════════════════════════
    # CATEGORY 1 — Emergency Events
    # ══════════════════════════════════════════════════════════════════════════

    def detect_accident_collision(self, relations_df: pd.DataFrame,
                                   master_df: pd.DataFrame) -> List[Dict]:
        events, seen = [], set()

        # Signal (a): direct collision predicate
        for _, row in relations_df[relations_df["predicate"] == "Colliding"].iterrows():
            subj, obj = str(row["subject"]), str(row["object"])
            if not (_is_vehicle(subj) or _is_vehicle(obj)):
                continue
            pair = tuple(sorted([subj, obj]))
            if pair in seen:
                continue
            seen.add(pair)
            events.append({
                "timestamp":  float(row["timestamp"]),
                "event_type": "AccidentCollisionDetection",
                "confidence": min(0.98, float(row["confidence"]) + 0.05),
                "object":     f"{subj} vs {obj}",
            })

        # Signal (b): braking + approaching + was fast — all THREE required
        if "acceleration" in master_df.columns and "velocity" in master_df.columns:
            veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]
            braking_tids = set(
                veh[(veh["acceleration"] < self.BRAKE_DECEL_THRESH) &
                    (veh["velocity"] > self.BRAKE_MIN_SPEED)]["track_id"].unique()
            )

            def _trim_warmup(df, n=5):
                return df.groupby("track_id", group_keys=False).apply(
                    lambda g: g.sort_values("frame_idx").iloc[n:]
                )

            veh_trimmed = _trim_warmup(veh)
            fast_tids = set(
                veh_trimmed[veh_trimmed["velocity"] > 150.0]["track_id"].unique()
            )
            qualified = braking_tids & fast_tids

            
            for _, row in relations_df[relations_df["predicate"] == "Approaching"].iterrows():
                subj = str(row["subject"])
                obj  = str(row.get("object", ""))
                if not _is_vehicle(subj):
                    continue
                obj_rows = master_df[master_df["track_id"] == _tid_from_name(obj)]
                if not obj_rows.empty and float(obj_rows["velocity"].mean()) < 5.0:
                    continue
                try:
                    tid = int(subj.split("_")[-1])
                except ValueError:
                    continue
                if tid not in qualified:
                    continue

                # Direction check: subject must be heading toward the object
                subj_rows = master_df[master_df["track_id"] == tid].sort_values("frame_idx")
                if "vx" in subj_rows.columns and "vy" in subj_rows.columns:
                    recent = subj_rows.tail(5)
                    vx_s = float(recent["vx"].mean())
                    vy_s = float(recent["vy"].mean())

                    obj_rows_inst = master_df[master_df["track_id"] == _tid_from_name(obj)]
                    if not obj_rows_inst.empty and not subj_rows.empty:
                        cx_s = float(subj_rows.tail(1)[["x1","x2"]].mean(axis=1).values[0])
                        cy_s = float(subj_rows.tail(1)[["y1","y2"]].mean(axis=1).values[0])
                        cx_o = float(obj_rows_inst[["x1","x2"]].mean(axis=1).mean())
                        cy_o = float(obj_rows_inst[["y1","y2"]].mean(axis=1).mean())
                        dx, dy = cx_o - cx_s, cy_o - cy_s
                        norm_s = (vx_s**2 + vy_s**2)**0.5
                        norm_d = (dx**2 + dy**2)**0.5
                        if norm_s > 0 and norm_d > 0:
                            cos_angle = (vx_s*dx + vy_s*dy) / (norm_s * norm_d)
                            if cos_angle < 0.5:  # not heading toward the object
                                continue

                pair = ("brake_fast_approach", subj)
                if pair in seen:
                    continue
                seen.add(pair)
                events.append({
                    "timestamp":  float(row["timestamp"]),
                    "event_type": "AccidentCollisionDetection",
                    "confidence": 0.80,
                    "object":     f"{subj} (braking-fast-approach)",
                })

        return self._validate(events)

    def detect_emergency_vehicle_tracking(self, master_df: pd.DataFrame,
                                           audio_tokens: List[Dict]) -> List[Dict]:
        events = []
        if "is_emergency" in master_df.columns:
            em_df = master_df[master_df["is_emergency"].astype(bool)]
        else:
            em_df = master_df[master_df["class_name"].isin(
                {"ambulance", "police", "fire_truck"})]

        siren_tss = [
            (t["start_time"] + t["end_time"]) / 2
            for t in audio_tokens
            if t.get("type") == "emergency"
            and float(t.get("confidence", 0)) >= 0.30
        ]

        for tid in em_df["track_id"].unique():
            rows     = em_df[em_df["track_id"] == tid]
            first_ts = float(rows["timestamp"].min())
            conf     = float(rows["confidence"].mean())

            audio_corr = any(abs(sts - first_ts) < 1.0 for sts in siren_tss)

            # Require siren OR high visual confidence — not just class name
            if not audio_corr and conf < 0.80:
                continue

            final_conf = min(1.0, conf + (0.10 if audio_corr else 0.0))
            events.append({
                "timestamp":  first_ts,
                "event_type": "EmergencyVehicleTracking",
                "confidence": final_conf,
                "object":     f"track_{tid}_{rows.iloc[0]['class_name']}",
            })

        return self._validate(events)

    def detect_stalled_broken_down(self, master_df: pd.DataFrame) -> List[Dict]:
        events = []
        MIN_FRAMES = 15   

        veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]
        # Which frames have other moving vehicles?
        moving_frames = set(
            veh[veh["velocity"] > 8.0]["frame_idx"].tolist()
        )

        for tid in veh["track_id"].unique():
            rows = veh[veh["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < MIN_FRAMES:
                continue
            if float(rows["velocity"].mean()) >= self.SPEED_STALL:
                continue
            if float(rows["velocity"].max()) > 8.0:
                continue

            # Exclusion: must overlap with frames where other cars move
            frames_here = set(rows["frame_idx"].tolist())
            if not frames_here.intersection(moving_frames):
                continue  # nobody else moving → not a road-blocking stall

            events.append({
                "timestamp":  float(rows["timestamp"].min()),
                "event_type": "StalledBrokenDownVehicle",
                "confidence": 0.80,
                "object":     f"{rows.iloc[0]['class_name']}_{tid}",
            })

        return self._validate(events)

    def detect_medical_emergency(self, master_df: pd.DataFrame,
                                  audio_tokens: List[Dict],scene_type: str = "vehicle_road") -> List[Dict]:
        events, seen_tids = [], set()

        # Scene-adaptive frame and velocity thresholds
        if scene_type == "pedestrian":
            MIN_FRAMES      = 25   # people wait long at metro/stations
            STILL_VEL       = 2.0  # truly motionless, not just slow walking
        elif scene_type == "mixed":
            MIN_FRAMES      = 18
            STILL_VEL       = 3.0
        else:  # vehicle_road
            MIN_FRAMES      = 12
            STILL_VEL       = self.MEDICAL_FALL_SPEED  

        persons = master_df[master_df["class_name"] == "person"]
        distress_ts = [
            (t["start_time"] + t["end_time"]) / 2
            for t in audio_tokens
            if any(kw in t["label"].lower() for kw in self.MEDICAL_AUDIO_KW)
            and float(t.get("confidence", 0)) >= 0.20
        ]

        for tid in persons["track_id"].unique():
            rows = persons[persons["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < MIN_FRAMES:
                continue

            # Use scene-adaptive STILL_VEL instead of hardcoded 5.0
            still_rows = rows[rows["velocity"] < STILL_VEL]
            if len(still_rows) < MIN_FRAMES:
                continue
            if tid in seen_tids:
                continue

            first_ts = float(still_rows["timestamp"].min())

            has_audio   = any(abs(ats - first_ts) < 1.5 for ats in distress_ts)
            has_cluster = False

            cx = float((still_rows.get("bbox_x",
                        (still_rows["x1"] + still_rows["x2"]) / 2)).mean()
                       if "bbox_x" in still_rows.columns
                       else ((still_rows["x1"] + still_rows["x2"]) / 2).mean())
            cy = float((still_rows.get("bbox_y",
                        (still_rows["y1"] + still_rows["y2"]) / 2)).mean()
                       if "bbox_y" in still_rows.columns
                       else ((still_rows["y1"] + still_rows["y2"]) / 2).mean())

            nearby = master_df[
                (master_df["class_name"] == "person") &
                (master_df["track_id"] != tid) &
                (abs(master_df["timestamp"] - first_ts) < 1.0)
            ]
            if not nearby.empty:
                nx = nearby.get("bbox_x", (nearby["x1"] + nearby["x2"]) / 2)
                ny = nearby.get("bbox_y", (nearby["y1"] + nearby["y2"]) / 2)
                dists = np.hypot(nx - cx, ny - cy)

                # ← Also require nearby persons to be stationary (not attacker/passerby)
                nearby_stationary = nearby[
                    (dists < 60.0) &
                    (nearby["velocity"] < STILL_VEL)
                ]
                has_cluster = len(nearby_stationary) >= 2

            if not has_audio and not has_cluster:
                continue

            conf = 0.72 + (0.12 if has_audio else 0) + (0.08 if has_cluster else 0)
            seen_tids.add(tid)
            events.append({
                "timestamp":  first_ts,
                "event_type": "MedicalEmergency",
                "confidence": round(min(0.95, conf), 3),
                "object":     f"person_{tid}",
            })

        return self._validate(events)
        
    # ══════════════════════════════════════════════════════════════════════════
    # CATEGORY 2 — Traffic Violations
    # ══════════════════════════════════════════════════════════════════════════

    def detect_wrong_side_driving(self, master_df: pd.DataFrame) -> List[Dict]:
        events, seen_tids = [], set()
        TIGHT_COS = -0.85  

        veh = master_df[
            master_df["class_name"].isin(VEHICLE_CLASSES) &
            (master_df["velocity"] > self.WRONG_WAY_MIN_SPD)
        ].copy()

        if len(veh) < 3:
            return events

        mean_vx = float(veh["vx"].mean())
        mean_vy = float(veh["vy"].mean())
        flow_mag = float(np.hypot(mean_vx, mean_vy))
        if flow_mag < 1e-3:
            if len(veh["track_id"].unique()) < 3:
                return events
            
        flow_vec = np.array([mean_vx, mean_vy]) / flow_mag

        for tid in veh["track_id"].unique():
            rows = veh[veh["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < self.MIN_FRAMES_WRONG_WAY:  # duration gate
                continue

            # Compute per-frame cosine and require MOST frames to be wrong-way
            avg_vx = float(rows["vx"].mean())
            avg_vy = float(rows["vy"].mean())
            spd    = float(np.hypot(avg_vx, avg_vy))
            if spd < self.WRONG_WAY_MIN_SPD:
                continue

            track_vec = np.array([avg_vx, avg_vy]) / spd
            cos_sim   = float(np.dot(track_vec, flow_vec))

            if cos_sim < TIGHT_COS and tid not in seen_tids:
                seen_tids.add(tid)
                conf = round(min(0.95, 0.75 + abs(cos_sim) * 0.20), 3)
                events.append({
                    "timestamp":  float(rows["timestamp"].min()),
                    "event_type": "WrongSideDriving",
                    "confidence": conf,
                    "object":     f"{rows.iloc[0]['class_name']}_{tid}",
                })

        return self._validate(events)
        
    def detect_speeding(self, master_df: pd.DataFrame) -> List[Dict]:
        events, seen_tids = [], set()
        veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]

        for tid in veh["track_id"].unique():
            if tid in seen_tids:
                continue
            rows = veh[veh["track_id"] == tid].sort_values("frame_idx")
            if len(rows) > 5:
                rows = rows.iloc[5:]
            fast = rows["velocity"] > self.SPEED_FAST
            if fast.sum() < self.MIN_FRAMES_SPEEDING:
                continue
            fast_vals = fast.values.astype(int)
            max_run = max(
                (sum(1 for _ in g) for v, g in
                 __import__("itertools").groupby(fast_vals) if v),
                default=0
            )
            if max_run < self.MIN_FRAMES_SPEEDING:
                continue

            first_fast_row = rows[fast].iloc[0]
            seen_tids.add(tid)
            events.append({
                "timestamp":  float(first_fast_row["timestamp"]),
                "event_type": "Speeding",
                "confidence": 0.85,
                "object":     f"{first_fast_row['class_name']}_{tid}",
            })

        return self._validate(events)
    

    def detect_illegal_parking(self, master_df: pd.DataFrame,
                                frame_width:  int = 640,
                                frame_height: int = 640) -> List[Dict]:
        events, seen_tids = [], set()
        MIN_FRAMES = 20  # casual stops shouldn't fire

        x_lo, x_hi = frame_width  * 0.20, frame_width  * 0.80  # tighter zone
        y_lo, y_hi = frame_height * 0.15, frame_height * 0.75

        veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]
        moving_frames = set(veh[veh["velocity"] > self.SPEED_STALL]["frame_idx"].tolist())

        for tid in veh["track_id"].unique():
            rows = veh[veh["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < MIN_FRAMES:
                continue
            if float(rows["velocity"].mean()) > self.SPEED_STALL:
                continue

            cx = float(((rows["x1"] + rows["x2"]) / 2).mean())
            cy = float(((rows["y1"] + rows["y2"]) / 2).mean())

            if not ((x_lo < cx < x_hi) and (y_lo < cy < y_hi)):
                continue

            frames_here = set(rows["frame_idx"].tolist())
            if not frames_here.intersection(moving_frames):
                continue

            if tid not in seen_tids:
                seen_tids.add(tid)
                events.append({
                    "timestamp":  float(rows["timestamp"].min()),
                    "event_type": "IllegalParking",
                    "confidence": 0.72,
                    "object":     f"{rows.iloc[0]['class_name']}_{tid}",
                })

        return self._validate(events)

    def detect_lane_discipline_violation(self, master_df: pd.DataFrame,
                                          fps: float = 25.0) -> List[Dict]:
        events, seen_tids = [], set()

        veh = master_df[
            master_df["class_name"].isin(VEHICLE_CLASSES) &
            (master_df["velocity"] > self.BRAKE_MIN_SPEED)
        ].copy()

        if veh.empty or "vx" not in veh.columns:
            return events

        veh["lat_ratio"] = veh["vx"].abs() / veh["velocity"].clip(lower=1e-3)

        for tid in veh["track_id"].unique():
            rows = veh[veh["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < self.MIN_FRAMES_DISTRACTED:
                continue

            lat_spd = rows["vx"].abs()
            wobble = (
                (rows["lat_ratio"] > 0.3) &
                (lat_spd >= self.DISTRACTED_LATERAL_LOW) &
                (lat_spd <= self.DISTRACTED_LATERAL_HIGH)
            )
            if wobble.sum() < self.MIN_FRAMES_DISTRACTED:
                continue

            # Require actual direction reversals (not just lateral drift)
            vx_vals      = rows["vx"].values
            sign_changes = int(np.sum(np.diff(np.sign(vx_vals)) != 0))
            if sign_changes < 2:
                continue   # single drift, not weaving

            if tid not in seen_tids:
                seen_tids.add(tid)
                conf = round(min(0.88, 0.65 + wobble.mean() * 0.25), 3)
                events.append({
                    "timestamp":  float(rows["timestamp"].min()),
                    "event_type": "LaneDisciplineViolation",
                    "confidence": conf,
                    "object":     f"{rows.iloc[0]['class_name']}_{tid}",
                })

        return self._validate(events)

    def detect_traffic_signal_breach(self, master_df: pd.DataFrame,
                                      frame_height: int = 640) -> List[Dict]:
        events, seen_tids = [], set()
        stop_y = frame_height * self.REDLIGHT_STOP_FRAC

        veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]

        for tid in veh["track_id"].unique():
            track_rows = veh[veh["track_id"] == tid].sort_values("timestamp")

            # Count frames in stop zone above speed threshold
            in_zone_fast = track_rows[
                (track_rows.apply(
                    lambda r: float(r.get("bbox_y", (r["y1"]+r["y2"])/2)), axis=1
                ) > stop_y) &
                (track_rows["velocity"] > self.REDLIGHT_MIN_SPD)
            ]
            if len(in_zone_fast) < 3:  # must be in zone+fast for 3+ frames
                continue

            first_ts = float(in_zone_fast.iloc[0]["timestamp"])
            prev_rows = track_rows[track_rows["timestamp"] < first_ts]
            decelerated = (
                "acceleration" in prev_rows.columns and
                (prev_rows["acceleration"] < -10).any()
            )
            if decelerated:
                continue

            if tid not in seen_tids:
                seen_tids.add(tid)
                spd  = float(in_zone_fast["velocity"].mean())
                conf = round(min(0.92, 0.70 + (spd / self.REDLIGHT_MIN_SPD) * 0.08), 3)
                events.append({
                    "timestamp":  first_ts,
                    "event_type": "TrafficSignalSignBreach",
                    "confidence": conf,
                    "object":     f"{track_rows.iloc[0]['class_name']}_{tid}",
                })

        return self._validate(events)


    def detect_safety_gear_non_compliance(self, master_df: pd.DataFrame,
                                           audio_tokens: List[Dict]) -> List[Dict]:
        events, seen_tids = [], set()

        moto = master_df[master_df["class_name"] == "motorcycle"]
        if moto.empty:
            return events

        moto_audio_ts = [
            (t["start_time"] + t["end_time"]) / 2
            for t in audio_tokens
            if any(kw in t["label"].lower() for kw in self.HELMET_AUDIO_KW)
            and float(t.get("confidence", 0)) >= 0.30   
        ]

        # Only fire if audio confirms motorcycle
        if not moto_audio_ts:
            return []

        for tid in moto["track_id"].unique():
            rows     = moto[moto["track_id"] == tid]
            first_ts = float(rows["timestamp"].min())
            audio_corr = any(abs(ats - first_ts) < 1.0 for ats in moto_audio_ts)
            if not audio_corr:
                continue

            if tid not in seen_tids:
                seen_tids.add(tid)
                events.append({
                    "timestamp":  first_ts,
                    "event_type": "SafetyGearNonCompliance",
                    "confidence": 0.65,
                    "object":     f"motorcycle_{tid} (audio-confirmed)",
                })

        return self._validate(events)

    # ══════════════════════════════════════════════════════════════════════════
    # CATEGORY 3 — Security & Crime
    # ══════════════════════════════════════════════════════════════════════════

    def detect_vehicle_theft(self, master_df: pd.DataFrame) -> List[Dict]:
        events, seen_tids = [], set()

        veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)].copy()

        # Find frames where many vehicles stop simultaneously → traffic light
        traffic_light_frames: Set[int] = set()
        for fidx, grp in veh.groupby("frame_idx"):
            n_stopped = (grp["velocity"] < self.SPEED_STALL).sum()
            if n_stopped >= 3:   # 3+ cars stopped = likely red light
                traffic_light_frames.add(int(fidx))

        for tid in veh["track_id"].unique():
            rows = veh[veh["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < (self.THEFT_STATIONARY_FRAMES + self.MIN_FRAMES_THEFT_BURST):
                continue

            speeds = rows["velocity"].values
            fidxs  = rows["frame_idx"].values

            for i in range(self.THEFT_STATIONARY_FRAMES, len(speeds)):
                stat_window = speeds[i - self.THEFT_STATIONARY_FRAMES : i]
                if not (stat_window < self.SPEED_STALL).all():
                    continue
                # Check burst persists
                burst_end = min(i + self.MIN_FRAMES_THEFT_BURST, len(speeds))
                burst_window = speeds[i : burst_end]
                if not (burst_window > self.THEFT_SPEED_THRESH).all():
                    continue

                # Exclusion: was the stationary period during a traffic light?
                stat_fidxs = set(int(f) for f in fidxs[i - self.THEFT_STATIONARY_FRAMES : i])
                if stat_fidxs.intersection(traffic_light_frames):
                    continue  # this is just a red-light stop + green-light go

                if tid not in seen_tids:
                    seen_tids.add(tid)
                    events.append({
                        "timestamp":  float(rows.iloc[i]["timestamp"]),
                        "event_type": "VehicleTheft",
                        "confidence": 0.73,
                        "object":     f"{rows.iloc[0]['class_name']}_{tid}",
                    })
                break

        return self._validate(events)
        
    def detect_robbery_snatching(self, master_df: pd.DataFrame,
                              audio_tokens: List[Dict]) -> List[Dict]:
        events, seen_tids = [], set()

        persons = master_df[master_df["class_name"] == "person"]
        if persons.empty or "acceleration" not in master_df.columns:
            return events

        distress_ts = [
            (t["start_time"] + t["end_time"]) / 2
            for t in audio_tokens
            if any(kw in t["label"].lower() for kw in {"scream", "yell", "shout"})
            and float(t.get("confidence", 0)) >= 0.25
        ]
        has_audio = len(distress_ts) > 0

        for tid in persons["track_id"].unique():
            rows = persons[persons["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < (self.MIN_FRAMES_SNATCH + 1):
                continue

            accel_vals = rows["acceleration"].values
            vel_vals   = rows["velocity"].values

            for i in range(len(accel_vals) - self.MIN_FRAMES_SNATCH + 1):
                accel_window = accel_vals[i: i + self.MIN_FRAMES_SNATCH]
                vel_window   = vel_vals[i: i + self.MIN_FRAMES_SNATCH]

                # Require sustained acceleration spike
                if not (accel_window > self.SNATCHING_ACCEL_THRESH).all():
                    continue

                spike_ts = float(rows.iloc[i]["timestamp"])

                # Path A: Audio-corroborated (high confidence)
                if has_audio and any(abs(dts - spike_ts) < 1.5 for dts in distress_ts):
                    if tid not in seen_tids:
                        seen_tids.add(tid)
                        events.append({
                            "timestamp":  spike_ts,
                            "event_type": "RobberySnatching",
                            "confidence": 0.78,
                            "object":     f"person_{tid} (sprint+distress)",
                        })
                    break

                # Path B: Visual-only fallback — sprint near stationary victim
                # Requires: sprinting person + another stationary person nearby
                if not has_audio:
                    # Check if there's a stationary person nearby (potential victim)
                    sprint_spd = float(vel_window.mean())
                    if sprint_spd < 25.0:  # must be actually sprinting
                        continue

                    cx = float(rows.iloc[i].get("bbox_x",
                        (rows.iloc[i]["x1"] + rows.iloc[i]["x2"]) / 2))
                    cy = float(rows.iloc[i].get("bbox_y",
                        (rows.iloc[i]["y1"] + rows.iloc[i]["y2"]) / 2))

                    nearby_persons = master_df[
                        (master_df["class_name"] == "person") &
                        (master_df["track_id"] != tid) &
                        (abs(master_df["timestamp"] - spike_ts) < 0.5)
                    ]
                    if nearby_persons.empty:
                        continue

                    nx = nearby_persons.get("bbox_x",
                         (nearby_persons["x1"] + nearby_persons["x2"]) / 2)
                    ny = nearby_persons.get("bbox_y",
                         (nearby_persons["y1"] + nearby_persons["y2"]) / 2)
                    dists = np.hypot(nx - cx, ny - cy)
                    stationary_nearby = nearby_persons[
                        (dists < self.PERSON_OBJ_DIST_PX) &
                        (nearby_persons["velocity"] < 8.0)
                    ]

                    if not stationary_nearby.empty and tid not in seen_tids:
                        seen_tids.add(tid)
                        events.append({
                            "timestamp":  spike_ts,
                            "event_type": "RobberySnatching",
                            "confidence": 0.62,  # lower — no audio confirmation
                            "object":     f"person_{tid} (sprint+victim_nearby)",
                        })
                    break

        return self._validate(events)
    
    def detect_abandoned_object(self, master_df: pd.DataFrame) -> List[Dict]:
        events, seen_tids = [], set()
        MIN_FRAMES = 25  
        OWNER_WINDOW_SEC = 3.0  

        OBJECT_CLASSES = {"suitcase", "backpack", "handbag", "bag",
                          "sports ball", "umbrella", "bottle", "box"}

        # Path A: small objects
        objs = master_df[master_df["class_name"].isin(OBJECT_CLASSES)]
        for tid in objs["track_id"].unique():
            rows = objs[objs["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < MIN_FRAMES:
                continue

            last_ts = float(rows["timestamp"].max())
            cx = float(((rows["x1"] + rows["x2"]) / 2).mean())
            cy = float(((rows["y1"] + rows["y2"]) / 2).mean())

            recent = master_df[
                (master_df["class_name"] == "person") &
                (abs(master_df["timestamp"] - last_ts) < OWNER_WINDOW_SEC)
            ]
            if not recent.empty:
                rx = (recent["x1"] + recent["x2"]) / 2
                ry = (recent["y1"] + recent["y2"]) / 2
                if (np.hypot(rx - cx, ry - cy) < self.PERSON_OBJ_DIST_PX).any():
                    continue

            if tid not in seen_tids:
                seen_tids.add(tid)
                events.append({
                    "timestamp":  float(rows["timestamp"].min()),
                    "event_type": "AbandonedObject",
                    "confidence": 0.68,
                    "object":     f"{rows.iloc[0]['class_name']}_{tid}",
                })

        # Path B: stationary vehicles without nearby driver
        veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]
        for tid in veh["track_id"].unique():
            if tid in seen_tids:
                continue
            rows = veh[veh["track_id"] == tid].sort_values("frame_idx")
            if len(rows) < MIN_FRAMES:
                continue
            if float(rows["velocity"].mean()) > self.SPEED_STALL:
                continue

            last_ts = float(rows["timestamp"].max())
            cx = float(((rows["x1"] + rows["x2"]) / 2).mean())
            cy = float(((rows["y1"] + rows["y2"]) / 2).mean())

            recent_persons = master_df[
                (master_df["class_name"] == "person") &
                (abs(master_df["timestamp"] - last_ts) < OWNER_WINDOW_SEC)
            ]
            owner_present = False
            if not recent_persons.empty:
                rx = (recent_persons["x1"] + recent_persons["x2"]) / 2
                ry = (recent_persons["y1"] + recent_persons["y2"]) / 2
                owner_present = (np.hypot(rx - cx, ry - cy) < self.PERSON_OBJ_DIST_PX * 2).any()

            if not owner_present:
                seen_tids.add(tid)
                events.append({
                    "timestamp":  float(rows["timestamp"].min()),
                    "event_type": "AbandonedObject",
                    "confidence": 0.65,
                    "object":     f"{rows.iloc[0]['class_name']}_{tid} (vehicle, no driver)",
                })

        return self._validate(events)

    # ══════════════════════════════════════════════════════════════════════════
    # CATEGORY 4 — Operational & Environmental
    # ══════════════════════════════════════════════════════════════════════════

    def detect_traffic_congestion(self, relations_df: pd.DataFrame, master_df: pd.DataFrame) -> List[Dict]:
        near = relations_df[
            (relations_df["predicate"] == "Near") &
            relations_df["subject"].apply(_is_vehicle)
        ]

        # Require Near pairs to involve at least one MOVING vehicle
        moving_tids = set(
            master_df[
                (master_df["class_name"].isin(VEHICLE_CLASSES)) &
                (master_df["velocity"] > 5.0)
            ]["track_id"].unique()
        )
        def _tid(name):
            try: return int(name.split("_")[-1])
            except: return -1
        veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]
        permanently_parked_tids = set()
        for tid, grp in veh.groupby("track_id"):
            grp = grp.sort_values("frame_idx")
            if (float(grp["velocity"].max()) < 5.0 and
                    float((grp["velocity"] < self.SPEED_STALL).mean()) > 0.80):
                permanently_parked_tids.add(tid)

        # Drop Near pairs where BOTH endpoints are permanently parked
        near = near[
            ~(near["subject"].apply(lambda s: _tid(s) in permanently_parked_tids) &
              near["object"].apply(lambda o: _tid(o) in permanently_parked_tids))
        ]
        # ─────────────────────────────────────────────────────────────────────

        if near.empty:
            return []

        # Sustained density check ─────────
        near = near.copy()
        near["time_bin"] = (near["timestamp"] // 2.0).astype(int)  # 2s bins
        bins_with_pairs  = near.groupby("time_bin").size()
        sustained_bins   = int((bins_with_pairs >= 4).sum())  # bins with 4+ pairs

        if sustained_bins >= 3:  # congestion must persist 6+ seconds
            return self._validate([{
                "timestamp":  float(near["timestamp"].median()),
                "event_type": "TrafficCongestion",
                "confidence": min(0.95, 0.60 + sustained_bins * 0.02),
                "object":     "multiple_vehicles",
            }])
            near = near[
                near["subject"].apply(lambda s: _tid(s) in moving_tids) |
                near["object"].apply(lambda o: _tid(o) in moving_tids)
            ]
        
        if len(near) >= 120:   
            return self._validate([{
                "timestamp":  float(near["timestamp"].median()),
                "event_type": "TrafficCongestion",
                "confidence": min(0.95, 0.60 + len(near) * 0.001),
                "object":     "multiple_vehicles",
            }])
        return []
        
    def detect_crowd_monitoring(self, master_df: pd.DataFrame,
                                 audio_tokens: List[Dict]) -> List[Dict]:
        CROWD_AUDIO_KW = {"crowd", "chatter", "babble", "cheering", "applause",
                          "screaming", "yell", "shout", "riot", "booing"}
        events, seen_windows = [], set()

        crowd_toks = [t for t in audio_tokens
                      if any(kw in t["label"].lower() for kw in CROWD_AUDIO_KW)
                      and float(t.get("confidence", 0)) >= self.CROWD_AUDIO_THRESH]

        for tok in crowd_toks:
            ts_mid  = (tok["start_time"] + tok["end_time"]) / 2
            win_key = round(ts_mid, 0)
            if win_key in seen_windows:
                continue

            nearby = master_df[
                (master_df["class_name"] == "person") &
                (abs(master_df["timestamp"] - ts_mid) <= 1.0)
            ]
            n = int(nearby["track_id"].nunique())

            # Require BOTH audio AND at least the base person threshold
            if n >= self.CROWD_PERSON_THRESH:
                seen_windows.add(win_key)
                density = min(1.0, n / (self.CROWD_PERSON_THRESH * 2))
                conf    = round((density + float(tok["confidence"])) / 2, 3)
                events.append({
                    "timestamp":  ts_mid,
                    "event_type": "CrowdMonitoring",
                    "confidence": conf,
                    "object":     f"{n}_persons + audio:{tok['label']}",
                })

        # Vision-only
        if not events:
            for ts in master_df[master_df["class_name"] == "person"]["timestamp"].unique():
                win_key = round(float(ts), 0)
                if win_key in seen_windows:
                    continue
                nearby = master_df[
                    (master_df["class_name"] == "person") &
                    (abs(master_df["timestamp"] - ts) <= 0.5)
                ]
                n = int(nearby["track_id"].nunique())
                if n >= self.CROWD_PERSON_THRESH * 3:  
                    seen_windows.add(win_key)
                    events.append({
                        "timestamp":  float(ts),
                        "event_type": "CrowdMonitoring",
                        "confidence": round(min(0.80, 0.52 + n * 0.015), 3),
                        "object":     f"{n}_persons (vision-only)",
                    })

        return self._validate(events)

    def detect_infrastructure_damage(self, master_df: pd.DataFrame,
                                      audio_tokens: List[Dict]) -> List[Dict]:
        events, seen_windows = [], set()

        damage_toks = [t for t in audio_tokens
                       if t.get("type") == "damage_context"
                       or any(kw in t["label"].lower()
                               for kw in self.INFRA_AUDIO_KEYWORDS)]

        for tok in damage_toks:
            ts_mid  = (tok["start_time"] + tok["end_time"]) / 2
            win_key = round(ts_mid, 0)
            if win_key in seen_windows:
                continue
            conf = float(tok.get("confidence", 0.60))
            nearby_veh = master_df[
                (master_df["class_name"].isin(VEHICLE_CLASSES)) &
                (abs(master_df["timestamp"] - ts_mid) < 2.0) &
                (master_df["velocity"] < self.SPEED_STALL)
            ]
            if not nearby_veh.empty:
                conf = min(1.0, conf + 0.20)
            seen_windows.add(win_key)
            events.append({
                "timestamp":  ts_mid,
                "event_type": "InfrastructureDamage",
                "confidence": round(conf, 3),
                "object":     f"audio:{tok['label']}",
            })

        if "acceleration" in master_df.columns:
            veh = master_df[master_df["class_name"].isin(VEHICLE_CLASSES)]
            for tid in veh["track_id"].unique():
                rows = veh[veh["track_id"] == tid].sort_values("timestamp")
                if len(rows) < 8:
                    continue
                for i in range(1, len(rows)):
                    acc = float(rows.iloc[i]["acceleration"])
                    if acc < self.BRAKE_DECEL_THRESH * 1.8:  
                        post_rows = rows.iloc[i:]
                        if (len(post_rows) >= 5 and    
                                float(post_rows["velocity"].mean()) < self.SPEED_STALL):
                            ts      = float(rows.iloc[i]["timestamp"])
                            win_key = round(ts, 0)
                            if win_key not in seen_windows:
                                seen_windows.add(win_key)
                                pre_rows = rows.iloc[:i]
                                if float(pre_rows["velocity"].max()) < 10.0:
                                    continue 
                                events.append({
                                    "timestamp":  ts,
                                    "event_type": "InfrastructureDamage",
                                    "confidence": 0.65,
                                    "object":     f"{rows.iloc[0]['class_name']}_{tid} (crash-still)",
                                })
                        break

        return self._validate(events)

    # ══════════════════════════════════════════════════════════════════════════
    # AGGREGATION  — wider window, cross-detector mutex applied first
    # ══════════════════════════════════════════════════════════════════════════

    @staticmethod
    def aggregate(events_df: pd.DataFrame, window: float = 5.0) -> pd.DataFrame:
        """
        Wider default window (5s, was 1s) to collapse same-event detections
        from multiple consecutive frames into one span.
        """
        if events_df.empty:
            return events_df
        agg = []
        for etype in events_df["event_type"].unique():
            sub = events_df[events_df["event_type"] == etype].sort_values("timestamp")
            grp = [sub.iloc[0].to_dict()]
            for i in range(1, len(sub)):
                if float(sub.iloc[i]["timestamp"]) - float(grp[-1]["timestamp"]) < window:
                    grp.append(sub.iloc[i].to_dict())
                else:
                    agg.append({
                        "start_time": grp[0]["timestamp"],
                        "end_time":   grp[-1]["timestamp"],
                        "event_type": etype,
                        "object":     grp[0]["object"],
                        "confidence": round(float(np.mean([g["confidence"] for g in grp])), 3),
                    })
                    grp = [sub.iloc[i].to_dict()]
            agg.append({
                "start_time": grp[0]["timestamp"],
                "end_time":   grp[-1]["timestamp"],
                "event_type": etype,
                "object":     grp[0]["object"],
                "confidence": round(float(np.mean([g["confidence"] for g in grp])), 3),
            })
        return pd.DataFrame(agg).sort_values("start_time").reset_index(drop=True)
        
    def run_all(self, master_df, relations_df, audio_tokens,
                frame_height=640, frame_width=640, fps=25.0,
                scene_type="vehicle_road"):   

        print(f"Running all event detectors (scene_type={scene_type})...")
        all_ev = []

        print("  [Cat 1] Emergency Events...")
        all_ev += self.detect_accident_collision(relations_df, master_df)
        all_ev += self.detect_emergency_vehicle_tracking(master_df, audio_tokens)
        # Skip stalled vehicle detection in pure pedestrian scenes — no vehicles
        if scene_type != "pedestrian":
            all_ev += self.detect_stalled_broken_down(master_df)
        all_ev += self.detect_medical_emergency(master_df, audio_tokens, scene_type)

        print("  [Cat 2] Traffic Violations...")
        if scene_type != "pedestrian":
            all_ev += self.detect_wrong_side_driving(master_df)
            all_ev += self.detect_speeding(master_df)
            all_ev += self.detect_illegal_parking(master_df, frame_width, frame_height)
            all_ev += self.detect_lane_discipline_violation(master_df, fps)
            all_ev += self.detect_traffic_signal_breach(master_df, frame_height)
            all_ev += self.detect_safety_gear_non_compliance(master_df, audio_tokens)

        print("  [Cat 3] Security & Crime...")
        if scene_type != "pedestrian":
            all_ev += self.detect_vehicle_theft(master_df)
        all_ev += self.detect_robbery_snatching(master_df, audio_tokens)
        all_ev += self.detect_abandoned_object(master_df)

        print("  [Cat 4] Operational & Environmental...")
        all_ev += self.detect_traffic_congestion(relations_df,master_df)
        all_ev += self.detect_crowd_monitoring(master_df, audio_tokens)
        all_ev += self.detect_infrastructure_damage(master_df, audio_tokens)

        if not all_ev:
            print("No events detected.")
            return pd.DataFrame(columns=["start_time", "end_time",
                                         "event_type", "confidence", "object"])

        all_ev = apply_mutex(all_ev)
        raw_df    = pd.DataFrame(all_ev)
        events_df = self.aggregate(raw_df, window=5.0)

        print(f"Done → {len(events_df)} aggregated events")
        print(events_df[["start_time", "event_type", "confidence",
                          "object"]].to_string(index=False))
        return events_df

m5        = M5EventEngine()
events_df = m5.run_all(
    master_df,
    relations_df,
    audio_tokens,
    frame_height = video_meta.height,
    frame_width  = video_meta.width,
    fps          = fps,
    scene_type   = scene_type_detected
)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell 14: M6 — Summarizer                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import torch, pandas as pd, re
from typing import List, Dict


class M6Summarizer:
    """
    Generates urban scene narrative via Phi-3 Mini (4-bit).
    Auto-falls back to a structured template if VRAM is insufficient.
    """
    MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

    def __init__(self):
        self._pipe   = None
        self._loaded = False
        print(" Summariser ready (lazy-loading Phi-3 Mini)")

    def _load(self):
        if self._loaded: return
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
            print(" Loading Phi-3 Mini (4-bit)…")
            tok = AutoTokenizer.from_pretrained(self.MODEL_ID, trust_remote_code=True)
            mdl = AutoModelForCausalLM.from_pretrained(
                self.MODEL_ID, device_map="auto", load_in_4bit=True,
                trust_remote_code=True, torch_dtype=torch.float16)
            self._pipe   = pipeline("text-generation", model=mdl, tokenizer=tok,batch_size=4)
            self._loaded = True
            print(" Phi-3 Mini loaded ✓")
        except Exception as e:
            print(f"[M6] Phi-3 load failed: {e}\n[M6] Using template fallback.")

    @staticmethod
    def _fmt_duration(sec: float) -> str:
        """Convert total seconds to natural human-readable duration."""
        sec = max(0.0, sec)
        if sec < 60:
            return f"{sec:.1f} seconds"
        elif sec < 3600:
            mins = int(sec // 60)
            secs = sec % 60
            if secs < 0.5:
                return f"{mins} minute{'s' if mins > 1 else ''}"
            return f"{mins} minute{'s' if mins > 1 else ''} {secs:.0f}s"
        else:
            hrs  = int(sec // 3600)
            mins = int((sec % 3600) // 60)
            if mins == 0:
                return f"{hrs} hour{'s' if hrs > 1 else ''}"
            return f"{hrs} hour{'s' if hrs > 1 else ''} {mins} min"

    @staticmethod
    def _fmt_ts(sec: float) -> str:
        """Format a single timestamp as a readable time reference."""
        sec = max(0.0, sec)
        if sec < 60:
            return f"{sec:.1f}s"
        elif sec < 3600:
            mins = int(sec // 60)
            secs = sec % 60
            if secs < 0.5:
                return f"{mins}min"
            return f"{mins}min {secs:.0f}s"
        else:
            hrs  = int(sec // 3600)
            mins = int((sec % 3600) // 60)
            return f"{hrs}hr {mins}min"

    @staticmethod
    def _clean_entity(obj: str) -> str:
        """
        Preserve entity ID format like car_1, person_9, truck_19.
        Only cleans up extra suffixes like '@0.033' from KG node labels.
        """
        return obj.split("@")[0].strip()

    
    @staticmethod
    def _build_prompt(triplets: List[Dict], events_df: pd.DataFrame,
                      stats: Dict, mode: str, duration_sec: float = 0.0) -> str:

        dur_str    = M6Summarizer._fmt_duration(duration_sec)
        mode_label = "Audio-Visual (AV)" if mode == "AV" else "Vision-Only"

        # ── Filter out uninformative predicates before sending to LLM ────────
        INFORMATIVE_PREDS = {
            "Colliding", "Approaching", "Overtaking", "Following",
            "HasStatus_Emergency", "EmergencyYielding", "EmergencySound",
            "DamageContextSound", "NearMiss",
        }
        informative = [t for t in triplets if t.get("predicate") in INFORMATIVE_PREDS]
        # Fall back to top by confidence if nothing informative found
        if not informative:
            informative = sorted(triplets, key=lambda x: -x.get("confidence", 0))[:8]
        top_t = informative[:12]

        t_lines = "\n".join(
            f"  [{t['predicate']}]  "
            f"{M6Summarizer._clean_entity(t['subject'])}  →  "
            f"{M6Summarizer._clean_entity(t['object'])}"
            for t in top_t
        ) if top_t else "  (no significant interactions detected)"

        # ── Events: proper timestamps, entity IDs preserved ───────────────────
        if events_df.empty:
            e_lines = "  (none detected)"
        else:
            e_lines = "\n".join(
                f"  [{r['event_type']}]  "
                f"{M6Summarizer._clean_entity(str(r['object']))}  "
                    f"at {M6Summarizer._fmt_ts(float(r['start_time']))}"
                for _, r in events_df.iterrows()
            )

        return (
            f"<|system|>\n"
            f"You are an urban traffic analyst. Write a concise factual "
            f"third-person narrative (3-5 sentences) from ONLY the data below. "
            f"Only describe what is explicitly listed. "
            f"Mention entities by their exact ID (e.g. car_1, person_9). "
            f"Use the timestamps exactly as given — do NOT convert or recalculate them. "
            f"If no significant events or interactions are listed, say so plainly. "
            f"Do NOT fabricate details not present in the data.\n"
            f"<|end|>\n<|user|>\n"
            f"Mode    : {mode_label}\n"
            f"Duration: {dur_str}\n"
            f"Stats   : {stats}\n\n"
            f"Key Interactions (significant predicates only):\n{t_lines}\n\n"
            f"Detected Events (timestamps already formatted):\n{e_lines}\n"
            f"<|end|>\n<|assistant|>"
        )

    # ── Template fallback ─────────────────────────────────────────────────────

    @staticmethod
    def _template(triplets: List[Dict], events_df: pd.DataFrame,
                  stats: Dict, mode: str, duration_sec: float = 0.0) -> str:
        n_t    = stats.get("unique_tracks", 0)
        n_v    = stats.get("vehicle_tracks", 0)
        n_p    = stats.get("person_tracks", 0)
        has_em = stats.get("emergency_audio", False)
        evs    = list(events_df["event_type"].unique()) if not events_df.empty else []
        sp_r   = [t for t in triplets if t["predicate"] in
                  ("Colliding", "Approaching", "Near", "Following", "Overtaking")][:3]

        dur_str    = M6Summarizer._fmt_duration(duration_sec)
        mode_label = "Audio-Visual (AV)" if mode == "AV" else "Vision-Only"

        s = (f"The scene ({mode_label} mode, {dur_str}) captured {n_t} unique objects "
             f"including {n_v} vehicle(s) and {n_p} pedestrian(s). ")
        if has_em:
            s += "Emergency audio (siren/horn) was detected during the clip. "
        if evs:
            s += f"Key events: {', '.join(set(evs))}. "
        if sp_r:
            s += "Observed interactions: " + "; ".join(
                f"{M6Summarizer._clean_entity(t['subject'])} "
                f"{t['predicate'].lower()} "
                f"{M6Summarizer._clean_entity(t['object'])}"
                for t in sp_r) + "."
        return s.strip()

    # ── Main generate ─────────────────────────────────────────────────────────

    def generate(self, kg_obj, events_df: pd.DataFrame,
                 stats: Dict, mode: str,
                 duration_sec: float = 0.0,
                 max_new_tokens: int = 220) -> str:
        self._load()
        triplets = (kg_obj.get_active_triplets()
                    if hasattr(kg_obj, "get_active_triplets") else [])

        if not triplets:
            return self._template([], events_df, stats, mode, duration_sec)
        if not self._loaded:
            return self._template(triplets, events_df, stats, mode, duration_sec)

        prompt = self._build_prompt(triplets, events_df, stats, mode, duration_sec)
        try:
            out = self._pipe(prompt, max_new_tokens=max_new_tokens,
                             do_sample=False, repetition_penalty=1.1)
            txt = out[0]["generated_text"]
            return (txt.split("<|assistant|>")[-1].strip()
                    if "<|assistant|>" in txt
                    else txt.split(prompt)[-1].strip())
        except Exception as e:
            print(f"[M6] Generation error: {e} — using template")
            return self._template(triplets, events_df, stats, mode, duration_sec)


# ── EXECUTION ─────────────────────────────────────────────────────────────────
m6        = M6Summarizer()
narrative = m6.generate(
    m4, events_df, scene_stats, pipeline_mode,
    duration_sec=video_meta.duration    
)

print("\n" + "="*60)
print("  URBAN SCENE NARRATIVE")
print("="*60)
print(narrative)
print("="*60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell 15: M7 — Visualizer                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np
import cv2
import os
from typing import Optional


class M7Visualizer:

    NODE_COLORS = {
        "object":      "#AED6F1",
        "audio_event": "#F1948A",
        "scene":       "#A9DFBF",
    }
    CRITICAL_PREDS = frozenset({
        "Colliding","Approaching","EmergencyYielding",
        "HasStatus_Emergency","Overtaking","NearMiss",
    })
    TRACK_COLORS = [
        (255,80,80),(80,200,80),(80,80,255),(220,180,0),
        (200,0,200),(0,190,190),(200,120,50),(100,50,200),
    ]

    def __init__(self): print(" Visualizer initialised")

    def plot_dkg(self, G: nx.MultiDiGraph, title: str = "Dynamic Knowledge Graph",
                 max_nodes: int = 100, figsize=(18,12), small_export=False):
        if G.number_of_nodes() == 0:
            print(" Graph empty — nothing to plot"); return

        nodes = list(G.nodes())
        if len(nodes) > max_nodes:
            deg   = dict(G.degree())
            nodes = sorted(nodes, key=lambda n: -deg[n])[:max_nodes]
            G     = G.subgraph(nodes).copy()

        fig, ax = plt.subplots(figsize=figsize)
        pos = nx.spring_layout(G, k=2.5, iterations=80, seed=42)

        if small_export:
            node_size  = 120
            font_size  = 4
            arrow_size = 5
            edge_width = 0.4
            legend_fs  = 4
            title_fs   = 6
            edge_label_fs = 3
        else:
            node_size  = 1800
            font_size  = 8
            arrow_size = 20
            edge_width = 1.5
            legend_fs  = 9
            title_fs   = 15
            edge_label_fs = 7

        node_colors = [self.NODE_COLORS.get(G.nodes[n].get("type","object"),"#D5D8DC")
                       for n in G.nodes()]
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_size,
                               edgecolors="#2C3E50", linewidths=0.4, alpha=0.9, ax=ax)
        nx.draw_networkx_labels(G, pos,
                                labels={n:str(G.nodes[n].get("label",n))[:20] for n in G.nodes()},
                                font_size=font_size, font_weight="bold", ax=ax)

        edge_colors = ["#E74C3C" if d.get("predicate","") in self.CRITICAL_PREDS
                       else "#AAB7B8" for u,v,d in G.edges(data=True)]
        nx.draw_networkx_edges(G, pos, edge_color=edge_colors,
                               arrowstyle="-|>", arrowsize=arrow_size, width=edge_width, alpha=0.8,
                               connectionstyle="arc3,rad=0.15", ax=ax)

        crit_labels = {}
        for u,v,d in G.edges(data=True):
            if d.get("predicate","") in self.CRITICAL_PREDS:
                crit_labels[(u,v)] = f"{d['predicate']}\n({d['confidence']:.2f})"
        nx.draw_networkx_edge_labels(G, pos, edge_labels=crit_labels,
                                      font_size=edge_label_fs, font_color="#922B21",
                                      bbox=dict(facecolor="white",alpha=0.6,edgecolor="none"),ax=ax)

        patches = [mpatches.Patch(color=c, label=t.replace("_"," ").title())
                   for t,c in self.NODE_COLORS.items()]
        patches.append(mpatches.Patch(color="#E74C3C", label="Critical Edge"))
        ax.legend(handles=patches, loc="upper left", fontsize=legend_fs)
        ax.set_title(title, fontsize=title_fs, fontweight="bold"); ax.axis("off")
        plt.tight_layout()
        out = os.path.join(WORKING_DIR, "dkg_full.png")
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
        print(f" DKG saved → {out}")

    # ── Semantic (pruned) DKG ────────────────────────────────────────────────
    def plot_semantic_dkg(self, G: nx.MultiDiGraph):
        Gp = G.copy()
        rm = [(u,v,k) for u,v,k,d in Gp.edges(keys=True,data=True)
              if d.get("predicate","") not in self.CRITICAL_PREDS]
        Gp.remove_edges_from(rm)
        Gp.remove_nodes_from(list(nx.isolates(Gp)))
        if Gp.number_of_nodes() == 0:
            print(" No critical-predicate edges — skipping semantic view."); return

        nodes = list(Gp.nodes())
        if len(nodes) > 60:
            deg   = dict(Gp.degree())
            nodes = sorted(nodes, key=lambda n: -deg[n])[:60]
            Gp    = Gp.subgraph(nodes).copy()

        fig, ax = plt.subplots(figsize=(8.5/2.54, 5.5/2.54))
        pos = nx.spring_layout(Gp, k=2.5, iterations=80, seed=42)

        node_colors = [self.NODE_COLORS.get(Gp.nodes[n].get("type","object"),"#D5D8DC")
                       for n in Gp.nodes()]
        nx.draw_networkx_nodes(Gp, pos, node_color=node_colors, node_size=120,
                               edgecolors="#2C3E50", linewidths=0.4, alpha=0.9, ax=ax)
        nx.draw_networkx_labels(Gp, pos,
                                labels={n:str(Gp.nodes[n].get("label",n))[:20] for n in Gp.nodes()},
                                font_size=2, font_weight="bold",font_family="monospace", ax=ax)

        edge_colors = ["#E74C3C" if d.get("predicate","") in self.CRITICAL_PREDS
                       else "#AAB7B8" for u,v,d in Gp.edges(data=True)]
        nx.draw_networkx_edges(Gp, pos, edge_color=edge_colors,
                               arrowstyle="-|>", arrowsize=5, width=0.4, alpha=0.8,
                               connectionstyle="arc3,rad=0.15", ax=ax)

        crit_labels = {}
        for u,v,d in Gp.edges(data=True):
            if d.get("predicate","") in self.CRITICAL_PREDS:
                crit_labels[(u,v)] = f"{d['predicate']}\n({d['confidence']:.2f})"
        nx.draw_networkx_edge_labels(Gp, pos, edge_labels=crit_labels,
                                      font_size=2, font_color="#922B21",
                                      bbox=dict(facecolor="white",edgecolor="none",pad=0),ax=ax)

        patches = [mpatches.Patch(color=c, label=t.replace("_"," ").title())
                   for t,c in self.NODE_COLORS.items()]
        patches.append(mpatches.Patch(color="#E74C3C", label="Critical Edge"))
        ax.legend(handles=patches, loc="upper left", fontsize=2)
        
        ax.set_title("Semantic DKG — Critical Predicates Only", fontsize=6, fontweight="bold")
        ax.axis("off")
        plt.tight_layout()

        out = os.path.join(WORKING_DIR, "dkg_full.webp")
        fig.set_size_inches(8.5/2.54, 5.5/2.54)
        plt.savefig(out, dpi=300, bbox_inches="tight", format="webp")
        plt.show()
        print(f" Semantic DKG saved → {out}")

    def render_annotated_video(self, video_path: str,
                                master_df: pd.DataFrame,
                                relations_df: pd.DataFrame,
                                max_frames: int = MAX_FRAMES) -> str:
        out = os.path.join(WORKING_DIR, "annotated_output.mp4")
        cap = cv2.VideoCapture(video_path)
        fps_v = cap.get(cv2.CAP_PROP_FPS) or 25.0
        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), fps_v, (W, H))

        df           = master_df.copy()
        stable_class = get_stable_class(df)   
        frame_index  = build_frame_index(df)  

        frame_idx = 0
        while cap.isOpened() and frame_idx < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            ts = round(frame_idx / fps_v, 3)

            ft = frame_index.get(frame_idx, pd.DataFrame())   
            fr = relations_df[abs(relations_df["timestamp"] - ts) < 0.1]

            for _, t in ft.iterrows():
                x1, y1, x2, y2 = get_bbox(t)                          
                tid = int(t["track_id"])
                is_critical = t.get("is_emergency", False)
                col = (0, 0, 255) if is_critical else (0, 200, 0)
                cls = stable_class.get(tid, str(t["class_name"]))      
                lbl = f"{cls} #{tid}" + ("!" if is_critical else "")
                cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
                cv2.putText(frame, lbl, (x1, max(y1 - 8, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)

                vx, vy = float(t.get("vx", 0)), float(t.get("vy", 0))
                spd = np.hypot(vx, vy)
                if spd > 5:
                    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                    sc = min(30, spd * 0.2)
                    cv2.arrowedLine(frame, (cx, cy),
                                    (int(cx + vx / spd * sc), int(cy + vy / spd * sc)),
                                    col, 2, tipLength=0.4)

            y_off = 25
            for _, r in fr[fr["predicate"].isin(self.CRITICAL_PREDS)].iterrows():
                txt = f"{r['subject']} --[{r['predicate']}]--> {r['object']}"
                cv2.putText(frame, txt, (10, y_off),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
                y_off += 18

            writer.write(frame)
            frame_idx += 1

        cap.release()
        writer.release()
        print(f" Annotated video → {out}  ({frame_idx} frames)")
        return out

    def export_pyvis(self, G: nx.MultiDiGraph,
                     out_html: Optional[str] = None):
        try: from pyvis.network import Network
        except ImportError:
            print(" pyvis not installed. Run: !pip install pyvis"); return
        out  = out_html or os.path.join(WORKING_DIR, "dkg_interactive.html")
        net  = Network(height="750px", width="100%", directed=True,
                       bgcolor="#1e1e2e", font_color="white")
        net.force_atlas_2based()
        for n,d in G.nodes(data=True):
            color = {"object":"#AED6F1","audio_event":"#F1948A","scene":"#A9DFBF"}.get(
                d.get("type","object"),"#AAAAAA")
            net.add_node(n, label=str(d.get("label",n))[:20], color=color, title=str(d))
        for u,v,d in G.edges(data=True):
            net.add_edge(u,v,label=d.get("predicate",""),
                         color="#E74C3C" if d.get("predicate","") in self.CRITICAL_PREDS else "#666666")
        net.write_html(out)
        print(f" Interactive DKG → {out}")


m7 = M7Visualizer()
m7.plot_dkg(KG, title=" Dynamic Knowledge Graph — Full View")
m7.plot_semantic_dkg(KG)
annotated_path = m7.render_annotated_video(OUTPUT_PATH, master_df, relations_df, max_frames=MAX_FRAMES)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell 16                                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import json, os
from IPython.display import Video, display

print("\n" + "╔" + "═"*60 + "╗")
print("║  MULTIMODAL DKG PIPELINE — REPORT" + " "*19 + "║")
print("╚" + "═"*60 + "╝")
print(f"  Video      : {os.path.basename(OUTPUT_PATH)}")
print(f"  Mode       : {pipeline_mode}")
print(f"  Duration   : {video_meta.duration:.1f}s @ {video_meta.fps:.1f} fps")
print(f"  Processed  : {MAX_FRAMES} frames")

print("\n──  Preprocessing ──────────────────────────────")
print(f"  Audio stream   : {video_meta.has_audio}")
print(f"  Audio tokens   : {len(audio_tokens)}")
print(f"  Emergency tokens: {sum(1 for t in audio_tokens if t.get('type')=='emergency')}")

print("\n──  Perception ─────────────────────────────────")
for k,v in scene_stats.items(): print(f"  {k}: {v}")

print("\n──  Tracking ───────────────────────────────────")
if not tracks_df.empty:
    print(f"  Unique tracks  : {tracks_df['track_id'].nunique()}")
    print(f"  Max speed      : {tracks_df['velocity'].max():.1f} px/s")

print("\n──  Fusion ────────────────────────────────────")
if not master_df.empty:
    em_count = master_df["is_emergency"].sum() if "is_emergency" in master_df.columns else 0
    print(f"  Emergency dets : {em_count}")
    print(f"  Class dist     : {master_df['class_name'].value_counts().to_dict()}")

print("\n──  Relations ──────────────────────────────────")
if not relations_df.empty:
    print(f"  Total triplets : {len(relations_df)}")
    print(f"  Predicates     : {relations_df['predicate'].value_counts().to_dict()}")

print("\n──  Knowledge Graph ────────────────────────────")
print(f"  Nodes          : {KG.number_of_nodes()}")
print(f"  Edges          : {KG.number_of_edges()}")

print("\n──  Events ─────────────────────────────────────")
if not events_df.empty:
    print(events_df[["start_time","event_type","confidence","object"]].to_string(index=False))
else:
    print("  (none)")

print("\n──  Narrative ──────────────────────────────────")
print(f"  {narrative}")

print("\n── Output Files ──────────────────────────────────")
print(f"  Annotated video  : {annotated_path}")
print(f"  DKG image        : {WORKING_DIR}/dkg_full.png")

report = {
    "video": os.path.basename(OUTPUT_PATH),
    "mode":  pipeline_mode,
    "scene_stats": scene_stats,
    "events": events_df.to_dict("records") if not events_df.empty else [],
    "narrative": narrative,
    "kg_nodes": KG.number_of_nodes(),
    "kg_edges": KG.number_of_edges(),
}
rp = os.path.join(WORKING_DIR, "pipeline_report.json")
with open(rp, "w") as f: json.dump(report, f, indent=2, default=str)
print(f"  JSON report      : {rp}")

print("\n── Annotated Video ───────────────────────────────")
display(Video(annotated_path, embed=True, width=640))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell 17: M8 — Event Response Suggestion                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, json, time, logging, heapq, datetime
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import IntEnum
 
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("M8")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 1.  PRIORITY LEVELS & EVENT MAPPING
# ══════════════════════════════════════════════════════════════════════════════
 
class Priority(IntEnum):
    CRITICAL = 1   # Immediate call (life-threatening)
    HIGH     = 2   # SMS alert + dashboard flag
    MEDIUM   = 3   # Dashboard flag only
    LOW      = 4   # Log only
 
# ── Full mapping: all 18 M5 event types ──────────────────────────────────────
EVENT_PRIORITY_MAP: Dict[str, Priority] = {
    # Category 1 — Emergency Events
    "AccidentCollisionDetection":  Priority.CRITICAL,
    "EmergencyVehicleTracking":    Priority.CRITICAL,
    "StalledBrokenDownVehicle":    Priority.MEDIUM,
    "MedicalEmergency":            Priority.CRITICAL,
 
    # Category 2 — Traffic Violations
    "WrongSideDriving":            Priority.HIGH,
    "Speeding":                    Priority.MEDIUM,
    "IllegalParking":              Priority.LOW,
    "LaneDisciplineViolation":     Priority.MEDIUM,
    "TrafficSignalSignBreach":     Priority.MEDIUM,
    "SafetyGearNonCompliance":     Priority.LOW,
 
    # Category 3 — Security & Crime
    "VehicleTheft":                Priority.HIGH,
    "RobberySnatching":            Priority.CRITICAL,
    "AbandonedObject":             Priority.HIGH,
 
    # Category 4 — Operational & Environmental
    "TrafficCongestion":           Priority.LOW,
    "CrowdMonitoring":             Priority.MEDIUM,
    "InfrastructureDamage":        Priority.HIGH,
}
 
DISPATCH_TARGET: Dict[Priority, str] = {
    Priority.CRITICAL: "Emergency Services (Ambulance / Fire / Police)",
    Priority.HIGH:     "Traffic Control Centre + Security",
    Priority.MEDIUM:   "Traffic Monitoring Dashboard",
    Priority.LOW:      "Incident Log / Archive",
}
 
# ── Call scripts for CRITICAL events only ─────────────────────────────────────
CALL_SCRIPTS: Dict[str, str] = {
    "AccidentCollisionDetection": (
        "URGENT: A vehicle collision has been detected at {location}. "
        "Object involved: {object_id}. Confidence: {confidence_pct}%. "
        "Dispatch ambulance and traffic control immediately."
    ),
    "EmergencyVehicleTracking": (
        "Alert: Emergency vehicle ({object_id}) detected at {location}. "
        "All units — clear the route immediately."
    ),
    "MedicalEmergency": (
        "URGENT: A possible medical emergency involving {object_id} "
        "at {location}. Confidence: {confidence_pct}%. Dispatch medical team."
    ),
    "RobberySnatching": (
        "SECURITY ALERT: Robbery or snatching detected. "
        "Subject: {object_id} at {location}. Confidence: {confidence_pct}%. "
        "Dispatch police immediately."
    ),
    "_default": (
        "ALERT: {event_type} detected at {location}. "
        "Object: {object_id}. Confidence: {confidence_pct}%. "
        "Immediate action required."
    ),
}
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 2.  ALERT RECORD
# ══════════════════════════════════════════════════════════════════════════════
 
@dataclass(order=True)
class AlertRecord:
    priority:     int
    timestamp:    float = field(compare=False)
    event_type:   str   = field(compare=False)
    object_id:    str   = field(compare=False)
    confidence:   float = field(compare=False)
    location:     str   = field(compare=False, default="Intersection_Unknown")
    top_feature:  str   = field(compare=False, default="N/A")
    action_taken: str   = field(compare=False, default="")
    suppressed:   bool  = field(compare=False, default=False)
    incident_id:  str   = field(compare=False, default="")   
 
    def to_dict(self) -> Dict:
        return {
            "priority":     Priority(self.priority).name,
            "timestamp":    round(self.timestamp, 3),
            "event_type":   self.event_type,
            "object_id":    self.object_id,
            "confidence":   self.confidence,
            "location":     self.location,
            "top_feature":  self.top_feature,
            "action_taken": self.action_taken,
            "suppressed":   self.suppressed,
            "incident_id":  self.incident_id,
        }
 
# ══════════════════════════════════════════════════════════════════════════════
# 3.  INCIDENT DEDUPLICATOR  
# ══════════════════════════════════════════════════════════════════════════════
 
class IncidentDeduplicator:
    """
    Prevents repeated calls/alerts for the same physical incident.
 
    An 'incident' is uniquely identified by:
      (event_type, object_id_normalised, time_bucket)
 
    Where time_bucket = floor(timestamp / WINDOW_SEC).
    Two events are the same incident if they share all three keys.
 
    For CRITICAL events specifically, a call is only placed ONCE per
    unique incident regardless of how many detection frames fire.
 
    Parameters
    ----------
    call_window_sec  : time window to deduplicate calls  (default 10s)
    alert_window_sec : time window to deduplicate alerts (default 5s)
    """
 
    def __init__(self,
                 call_window_sec:  float = 10.0,
                 alert_window_sec: float = 5.0):
        self.call_window   = call_window_sec
        self.alert_window  = alert_window_sec
        self._seen_calls:  Dict[str, float] = {}   
        self._seen_alerts: Dict[str, float] = {}   
 
    def _normalise_object(self, object_id: str) -> str:
        """
        Strip frame-specific suffixes to get a stable identity.
        e.g. 'car_3 vs truck_7' stays as-is;
             'person_6@0.480' → 'person_6'
        """
        return object_id.split("@")[0].strip()
 
    def get_incident_id(self, event_type: str,
                        object_id: str,
                        timestamp: float,
                        window: float) -> str:
        obj_norm  = self._normalise_object(object_id)
        time_slot = int(timestamp // window)
        return f"{event_type}|{obj_norm}|{time_slot}"
 
    def should_call(self, alert: AlertRecord) -> bool:
        """Returns True if a call should be placed (not a duplicate)."""
        iid = self.get_incident_id(
            alert.event_type, alert.object_id,
            alert.timestamp, self.call_window
        )
        if iid in self._seen_calls:
            return False
        self._seen_calls[iid] = alert.timestamp
        alert.incident_id = iid
        return True
 
    def should_alert(self, alert: AlertRecord) -> bool:
        """Returns True if an alert/SMS should be sent (not a duplicate)."""
        iid = self.get_incident_id(
            alert.event_type, alert.object_id,
            alert.timestamp, self.alert_window
        )
        if iid in self._seen_alerts:
            return False
        self._seen_alerts[iid] = alert.timestamp
        alert.incident_id = iid
        return True
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 4.  THRESHOLD TUNER
# ══════════════════════════════════════════════════════════════════════════════
 
class ThresholdTuner:
    """Per-event-type confidence thresholds; noise-augmented false-alarm suppression."""
 
    DEFAULT_THRESHOLDS: Dict[str, float] = {
        # Category 1
        "AccidentCollisionDetection":  0.55,
        "EmergencyVehicleTracking":    0.50,
        "StalledBrokenDownVehicle":    0.70,
        "MedicalEmergency":            0.55,
        # Category 2
        "WrongSideDriving":            0.60,
        "Speeding":                    0.70,
        "IllegalParking":              0.65,
        "LaneDisciplineViolation":     0.55,
        "TrafficSignalSignBreach":     0.60,
        "SafetyGearNonCompliance":     0.50,
        # Category 3
        "VehicleTheft":                0.60,
        "RobberySnatching":            0.55,
        "AbandonedObject":             0.58,
        # Category 4
        "TrafficCongestion":           0.55,
        "CrowdMonitoring":             0.50,
        "InfrastructureDamage":        0.55,
    }
 
    def __init__(self, noise_sigma: float = 0.05,
                 n_augments:  int   = 50,
                 margin:      float = 0.5,
                 seed:        int   = 42):
        self.noise_sigma = noise_sigma
        self.n_augments  = n_augments
        self.margin      = margin
        self.rng         = np.random.default_rng(seed)
        self.tuned: Dict[str, float] = dict(self.DEFAULT_THRESHOLDS)
 
    def fit(self, events_df: pd.DataFrame) -> Dict[str, float]:
        if events_df.empty:
            return self.tuned
        for etype, grp in events_df.groupby("event_type"):
            confs     = grp["confidence"].values.astype(float)
            noise     = self.rng.normal(0, self.noise_sigma,
                                        size=(len(confs), self.n_augments))
            augmented = np.clip((confs[:, None] + noise).flatten(), 0.0, 1.0)
            mu, sigma = augmented.mean(), augmented.std()
            self.tuned[etype] = round(float(np.clip(mu - self.margin * sigma, 0.10, 0.95)), 4)
        print("[M8-ThresholdTuner] Tuned thresholds:")
        for et, th in self.tuned.items():
            print(f"   {et:<40} → {th:.4f}")
        return self.tuned
 
    def is_above_threshold(self, event_type: str, confidence: float) -> bool:
        th = self.tuned.get(event_type,
             self.DEFAULT_THRESHOLDS.get(event_type, 0.50))
        return confidence >= th
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 5.  FEATURE ATTRIBUTOR
# ══════════════════════════════════════════════════════════════════════════════
 
class FeatureAttributor:
    FEATURE_MAP: Dict[str, List[str]] = {
        "AccidentCollisionDetection":  ["confidence", "velocity", "acceleration"],
        "EmergencyVehicleTracking":    ["is_emergency", "siren_prob", "confidence"],
        "StalledBrokenDownVehicle":    ["velocity", "confidence"],
        "MedicalEmergency":            ["velocity", "bbox_y", "confidence"],
        "WrongSideDriving":            ["vx", "vy", "velocity"],
        "Speeding":                    ["velocity", "confidence"],
        "IllegalParking":              ["velocity", "confidence"],
        "LaneDisciplineViolation":     ["vx", "velocity"],
        "TrafficSignalSignBreach":     ["velocity", "bbox_y"],
        "SafetyGearNonCompliance":     ["confidence"],
        "VehicleTheft":                ["velocity", "acceleration"],
        "RobberySnatching":            ["acceleration", "velocity"],
        "AbandonedObject":             ["confidence", "velocity"],
        "TrafficCongestion":           ["confidence"],
        "CrowdMonitoring":             ["confidence"],
        "InfrastructureDamage":        ["acceleration", "velocity"],
    }
 
    def attribute(self, alert: AlertRecord, master_df: pd.DataFrame) -> str:
        candidates = self.FEATURE_MAP.get(alert.event_type, ["confidence"])
        if master_df.empty:
            return f"{candidates[0]}={alert.confidence:.3f} (fallback)"
        window = master_df[abs(master_df["timestamp"] - alert.timestamp) < 1.0]
        best_feat, best_val = candidates[0], 0.0
        for feat in candidates:
            if feat in window.columns:
                vals = window[feat].dropna().abs()
                if len(vals) > 0:
                    v = float(vals.max())
                    if v > best_val:
                        best_val  = v
                        best_feat = feat
        return f"{best_feat}={best_val:.3f}"
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 6.  JSON CALL LOGGER
# ══════════════════════════════════════════════════════════════════════════════
 
class JSONCallLogger:
    """Logs calls to JSON. No Twilio / no real calls."""
 
    def __init__(self, working_dir: str = "/kaggle/working"):
        self.working_dir = working_dir
        self.call_log: List[Dict] = []
        self.log_path = os.path.join(working_dir, "call_dispatch_log.json")
        os.makedirs(working_dir, exist_ok=True)
 
    def _build_message(self, alert: AlertRecord) -> str:
        template = CALL_SCRIPTS.get(alert.event_type, CALL_SCRIPTS["_default"])
        return template.format(
            event_type     = alert.event_type,
            location       = alert.location,
            object_id      = alert.object_id,
            confidence_pct = int(alert.confidence * 100),
        )
 
    def _write_log(self):
        with open(self.log_path, "w") as f:
            json.dump(self.call_log, f, indent=2, default=str)
 
    def place_call(self, alert: AlertRecord) -> str:
        call_id = f"CALL_{int(time.time() * 1000)}"
        message = self._build_message(alert)
        utc_now = datetime.datetime.utcnow().isoformat()
 
        record = {
            "call_id":         call_id,
            "incident_id":     alert.incident_id,
            "event_type":      alert.event_type,
            "object_id":       alert.object_id,
            "location":        alert.location,
            "priority":        Priority(alert.priority).name,
            "confidence":      round(alert.confidence, 4),
            "top_feature":     alert.top_feature,
            "message":         message,
            "status":          "CALL_INITIATED",
            "utc_time":        utc_now,
            "video_timestamp": round(alert.timestamp, 3),
        }
        self.call_log.append(record)
        self._write_log()
 
        border = "─" * 64
        print(f"\n┌{border}┐")
        print(f"│   CALL INITIATED  [{utc_now[11:19]}]" + " " * 22 + "│")
        print(f"├{border}┤")
        print(f"│  Event      : {alert.event_type:<49}│")
        print(f"│  Object     : {alert.object_id:<49}│")
        print(f"│  Location   : {alert.location:<49}│")
        print(f"│  Conf       : {alert.confidence:<49.4f}│")
        print(f"│  IncidentID : {alert.incident_id:<49}│")
        print(f"│  CallID     : {call_id:<49}│")
        print(f"├{border}┤")
        words, line = message.split(), ""
        wrapped = []
        for w in words:
            if len(line) + len(w) + 1 > 60:
                wrapped.append(line); line = w
            else:
                line = (line + " " + w).strip()
        if line: wrapped.append(line)
        for wl in wrapped:
            print(f"│  {wl:<59}│")
        print(f"├{border}┤")
        print(f"│   Logged → call_dispatch_log.json" + " " * 26 + "│")
        print(f"└{border}┘\n")
 
        return call_id
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 7.  ALERT PRIORITY QUEUE
# ══════════════════════════════════════════════════════════════════════════════
 
class AlertPriorityQueue:
    def __init__(self):
        self._heap: List[AlertRecord] = []
 
    def push(self, alert: AlertRecord):
        heapq.heappush(self._heap, alert)
 
    def pop(self) -> AlertRecord:
        return heapq.heappop(self._heap)
 
    def __len__(self):
        return len(self._heap)
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 8.  LOCATION RESOLVER
# ══════════════════════════════════════════════════════════════════════════════
 
class LocationResolver:
    def __init__(self, video_name: str = "scene", n_zones: int = 4):
        self.zones = [f"Intersection_{i+1}" for i in range(n_zones)]
 
    def resolve(self, timestamp: float, duration: float) -> str:
        if duration <= 0:
            return self.zones[0]
        frac = min(1.0, timestamp / duration)
        return self.zones[int(frac * len(self.zones)) % len(self.zones)]
 
 
# ══════════════════════════════════════════════════════════════════════════════
# 9.  MAIN MODULE — M8ActionDeployment
# ══════════════════════════════════════════════════════════════════════════════
 
class M8ActionDeployment:
    """
    Orchestrates the full action & deployment pipeline with smart deduplication.
 
    Key behaviours
    ──────────────
    CRITICAL events  → ONE call per incident (deduplicated by event_type +
                        object + 10-second window). No repeated calls for the
                        same collision detected across multiple frames.
    HIGH events      → ONE SMS/alert per incident (5-second window dedup).
    MEDIUM events    → Dashboard flag raised once per incident.
    LOW events       → Logged once per incident.
    Suppressed       → Below noise threshold; silently dropped.
    """
 
    def __init__(self,
                 working_dir:  str   = "/kaggle/working",
                 noise_sigma:  float = 0.05,
                 n_augments:   int   = 50,
                 margin:       float = 0.5,
                 call_window:  float = 10.0,
                 alert_window: float = 5.0):
 
        self.working_dir = working_dir
        os.makedirs(working_dir, exist_ok=True)
 
        self.tuner       = ThresholdTuner(noise_sigma, n_augments, margin)
        self.attributor  = FeatureAttributor()
        self.call_logger = JSONCallLogger(working_dir)
        self.deduplicator= IncidentDeduplicator(call_window, alert_window)
        self.queue       = AlertPriorityQueue()
        self.loc_resolver= LocationResolver()
 
        self.dispatched:  List[AlertRecord] = []
        self.suppressed:  List[AlertRecord] = []
        self.deduped_skip: List[AlertRecord] = []   # valid but duplicate incidents
 
        print("[M8]  M8ActionDeployment initialised (smart dedup active).")
 
    # ── Internal helpers ──────────────────────────────────────────────────────
 
    def _make_alert(self, row: pd.Series, master_df: pd.DataFrame,
                    duration: float) -> AlertRecord:
        etype  = str(row["event_type"])
        conf   = float(row["confidence"])
        ts     = float(row.get("start_time", row.get("timestamp", 0.0)))
        obj_id = str(row.get("object", "unknown"))
 
        priority   = EVENT_PRIORITY_MAP.get(etype, Priority.LOW)
        location   = self.loc_resolver.resolve(ts, duration)
        suppressed = not self.tuner.is_above_threshold(etype, conf)
 
        alert = AlertRecord(
            priority    = int(priority),
            timestamp   = ts,
            event_type  = etype,
            object_id   = obj_id,
            confidence  = round(conf, 4),
            location    = location,
            suppressed  = suppressed,
        )
        alert.top_feature = self.attributor.attribute(alert, master_df)
        return alert
 
    def _dispatch_critical(self, alert: AlertRecord):
        """One call per unique incident only."""
        if self.deduplicator.should_call(alert):
            call_id = self.call_logger.place_call(alert)
            alert.action_taken = f"CALL_DISPATCHED (id={call_id})"
            log.info(" CRITICAL | %s @ %s | conf=%.2f | incident=%s",
                     alert.event_type, alert.location,
                     alert.confidence, alert.incident_id)
        else:
            alert.action_taken = "DUPLICATE_CALL_SUPPRESSED"
            self.deduped_skip.append(alert)
            log.info(" CRITICAL (dup suppressed) | %s | incident already active",
                     alert.event_type)
 
    def _dispatch_high(self, alert: AlertRecord):
        """One SMS/alert per unique incident."""
        if self.deduplicator.should_alert(alert):
            ts  = datetime.datetime.now().strftime("%H:%M:%S")
            msg = (f"[{ts}] !  HIGH ALERT — {alert.event_type} "
                   f"| {alert.object_id} @ {alert.location} "
                   f"| conf={alert.confidence:.2f} "
                   f"| feature={alert.top_feature}")
            print(msg)
            alert.action_taken = "SMS_ALERT_SENT (simulated)"
            log.warning("%s", msg)
        else:
            alert.action_taken = "DUPLICATE_ALERT_SUPPRESSED"
            self.deduped_skip.append(alert)
 
    def _dispatch_medium(self, alert: AlertRecord):
        if self.deduplicator.should_alert(alert):
            msg = (f"[MEDIUM] {alert.event_type} | {alert.object_id} "
                   f"@ {alert.location} | conf={alert.confidence:.2f} "
                   f"| feature={alert.top_feature}")
            print(f"   {msg}")
            alert.action_taken = "DASHBOARD_FLAGGED"
            log.info("%s", msg)
        else:
            alert.action_taken = "DUPLICATE_FLAG_SUPPRESSED"
            self.deduped_skip.append(alert)
 
    def _dispatch_low(self, alert: AlertRecord):
        if self.deduplicator.should_alert(alert):
            msg = (f"[LOW]  {alert.event_type} | {alert.object_id} "
                   f"@ {alert.location} | conf={alert.confidence:.2f}")
            print(f"   {msg}")
            alert.action_taken = "LOGGED"
            log.debug("%s", msg)
        else:
            alert.action_taken = "DUPLICATE_LOG_SUPPRESSED"
            self.deduped_skip.append(alert)
 
    def _process_queue(self):
        dispatch_map = {
            Priority.CRITICAL: self._dispatch_critical,
            Priority.HIGH:     self._dispatch_high,
            Priority.MEDIUM:   self._dispatch_medium,
            Priority.LOW:      self._dispatch_low,
        }
        print(f"\n[M8] Processing alert queue ({len(self.queue)} alerts)…")
        print("─" * 68)
        while self.queue:
            alert = self.queue.pop()
            dispatch_map[Priority(alert.priority)](alert)
            if alert.action_taken not in (
                "DUPLICATE_CALL_SUPPRESSED",
                "DUPLICATE_ALERT_SUPPRESSED",
                "DUPLICATE_FLAG_SUPPRESSED",
                "DUPLICATE_LOG_SUPPRESSED",
            ):
                self.dispatched.append(alert)
        print("─" * 68)
 
    def _save_report(self, narrative: str, events_df: pd.DataFrame) -> str:
        ts_str  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        outfile = os.path.join(self.working_dir, f"m8_report_{ts_str}.json")
        report  = {
            "generated_at":     datetime.datetime.utcnow().isoformat(),
            "narrative":        narrative,
            "tuned_thresholds": self.tuner.tuned,
            "total_events":     len(events_df),
            "dispatched":       [a.to_dict() for a in self.dispatched],
            "suppressed":       [a.to_dict() for a in self.suppressed],
            "deduped_skipped":  [a.to_dict() for a in self.deduped_skip],
            "call_log":         self.call_logger.call_log,
        }
        with open(outfile, "w") as f:
            json.dump(report, f, indent=2, default=str)
        print(f"\n[M8]   Post-incident report saved → {outfile}")
        return outfile
 
    def _print_summary(self):
        critical  = [a for a in self.dispatched if a.priority == Priority.CRITICAL]
        high      = [a for a in self.dispatched if a.priority == Priority.HIGH]
        medium    = [a for a in self.dispatched if a.priority == Priority.MEDIUM]
        low_      = [a for a in self.dispatched if a.priority == Priority.LOW]
        calls_placed = len(self.call_logger.call_log)
 
        border = "═" * 68
        print(f"\n╔{border}╗")
        print(f"║  M8 DEPLOYMENT SUMMARY" + " " * 45 + "║")
        print(f"╠{border}╣")
        print(f"║  {'Category':<22} {'Dispatched':>10}  {'Action':<32}║")
        print(f"╠{border}╣")
        print(f"║  {' CRITICAL':<22} {len(critical):>10}  {'Emergency call dispatched':<32}║")
        print(f"║  {'  HIGH':<22} {len(high):>10}  {'SMS alert sent':<32}║")
        print(f"║  {' MEDIUM':<22} {len(medium):>10}  {'Dashboard flag raised':<32}║")
        print(f"║  {' LOW':<22} {len(low_):>10}  {'Logged to archive':<32}║")
        print(f"║  {' Noise-suppressed':<22} {len(self.suppressed):>10}  {'Below threshold — dropped':<32}║")
        print(f"║  {' Dedup-suppressed':<22} {len(self.deduped_skip):>10}  {'Duplicate incident — skipped':<32}║")
        print(f"╠{border}╣")
        print(f"║   Calls actually placed : {calls_placed:<40}║")
        print(f"╚{border}╝")
 
        if critical:
            print("\n  Critical events actioned:")
            for a in critical:
                print(f"    • {a.event_type:<35} @ {a.location}"
                      f" | conf={a.confidence:.2f}"
                      f" | action={a.action_taken}")
 
        if self.call_logger.call_log:
            print(f"\n  ── Calls Placed ({calls_placed}) ──")
            for c in self.call_logger.call_log:
                print(f"     {c['call_id']} | {c['event_type']}"
                      f" | {c['location']} | {c['status']}")
 
    
    def run(self,
            events_df: pd.DataFrame,
            master_df: pd.DataFrame,
            narrative: str   = "",
            duration:  float = 0.0) -> List[AlertRecord]:
 
        print("\n" + "╔" + "═" * 62 + "╗")
        print("║  MODULE 8 — ACTION & DEPLOYMENT (Smart Dispatch)" + " " * 12 + " ║")
        print("╚" + "═" * 62 + "╝")
 
        if events_df.empty:
            print("[M8] No events to process. Exiting.")
            return []
 
        # Step 1 — Tune thresholds
        print("\n[M8] Step 1/5 — Threshold tuning…")
        self.tuner.fit(events_df)
 
        # Step 2 — Build and classify alerts
        print("\n[M8] Step 2/5 — Building alert records…")
        for _, row in events_df.iterrows():
            alert = self._make_alert(row, master_df, duration)
            if alert.suppressed:
                self.suppressed.append(alert)
                print(f"    SUPPRESSED  {alert.event_type:<35}"
                      f"conf={alert.confidence:.3f}")
            else:
                self.queue.push(alert)
 
        print(f"\n[M8] {len(self.queue)} alerts queued | "
              f"{len(self.suppressed)} noise-suppressed.")
 
        # Step 3 — Process queue (CRITICAL first, dedup active)
        print("\n[M8] Step 3/5 — Dispatching (dedup active)…")
        self._process_queue()
 
        # Step 4 — Summary
        print("\n[M8] Step 4/5 — Summary…")
        self._print_summary()
 
        # Step 5 — Save report
        print("\n[M8] Step 5/5 — Saving post-incident report…")
        self._save_report(narrative, events_df)
 
        return self.dispatched
 
 
# ── EXECUTION ──────────────────────────────────────────────────────────────────
m8 = M8ActionDeployment(
    working_dir  = WORKING_DIR,
    noise_sigma  = 0.05,
    n_augments   = 50,
    margin       = 0.5,
    call_window  = 10.0,   # same incident within 10s → only 1 call
    alert_window = 5.0,    # same incident within 5s  → only 1 alert/flag
)
 
dispatched_alerts = m8.run(
    events_df = events_df,
    master_df = master_df,
    narrative = narrative,
    duration  = video_meta.duration,
)

## Evaluation Metrics

The following 20 metrics assess pipeline quality across four dimensions:

**Audio-Visual Fusion**
- M1 Emergency Token Ratio — proportion of audio tokens classified as emergency type
- M2 AV Mode Activation — whether audio-visual fusion was triggered
- M5 Siren-Emergency Co-occurrence — P(emergency detection | siren detected)
- M12 AV vs Vision-Only Event Delta — additional events gained from audio fusion

**Tracking & Motion**
- M3 Mean Detection Confidence — average YOLO confidence across all detections
- M4 Velocity Smoothness — mean intra-track velocity std (lower = smoother tracks)
- M5 Mean Posterior Confidence + Fusion Gain — Bayesian update quality

**Knowledge Graph**
- M8 Node Coverage Ratio — fraction of M1 tracks represented in KG
- M9 Mean Edge Confidence — average confidence across KG edges
- M10 Prune Efficiency — fraction of stale edges correctly pruned
- M7 Predicate Diversity Entropy — richness of relation types in the graph

**Narrative Quality**
- M13 Entity Coverage — entity classes from KG mentioned in narrative
- M14 Entity Hallucination Rate — entities in narrative not grounded in KG/detections
- M15 Event Coverage — detected events reflected in narrative
- M16 Narrative Length — sentence count check (target: 3–5 sentences)
- M18 Class Recall — detected object classes mentioned in narrative
- M19 Compression Ratio — narrative length relative to KG triplet input
- M20 BERTScore — semantic similarity between narrative and KG triplets

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Cell 18: Evaluation Metrics                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import time, math, re
import numpy as np
import pandas as pd
from bert_score import score as bert_score_fn

print("\n" + "╔" + "═"*62 + "╗")
print("║  MULTIMODAL DKG PIPELINE — EVALUATION SCORECARD" + " "*13 + "║")
print("╚" + "═"*62 + "╝")

results = {}

# ─────────────────────────────────────────────────────────────
# Metric 1: Emergency Token Ratio
# ─────────────────────────────────────────────────────────────
total_tokens     = len(audio_tokens)
emergency_tokens = sum(1 for t in audio_tokens if t.get("type") == "emergency")
met1 = round(emergency_tokens / total_tokens, 4) if total_tokens > 0 else 0.0
results["1. Emergency Token Ratio"] = met1
print(f"\n  Emergency Token Ratio      : {met1:.4f}  "
      f"({emergency_tokens} emergency / {total_tokens} total tokens)")

# ─────────────────────────────────────────────────────────────
# Metric 2: AV Mode Activation
# ─────────────────────────────────────────────────────────────
met2 = 1.0 if pipeline_mode == "AV" else 0.0
results["2. AV Mode Activated"] = met2
print(f"  AV Mode Activated          : {met2}  (mode = '{pipeline_mode}')")

# ─────────────────────────────────────────────────────────────
# Metric 3: Mean Detection Confidence
# ─────────────────────────────────────────────────────────────
met3 = round(float(detections_df["confidence"].mean()), 4) if not detections_df.empty else 0.0
results["3. Mean Detection Confidence"] = met3
print(f"  Mean Detection Confidence  : {met3:.4f}  (threshold = 0.25)")

# ─────────────────────────────────────────────────────────────
# Metric 4: Velocity Smoothness
# ─────────────────────────────────────────────────────────────
if not tracks_df.empty:
    met4 = round(float(tracks_df.groupby("track_id")["velocity"].std().fillna(0).mean()), 4)
else:
    met4 = 0.0
results["4. Velocity Smoothness (mean intra-track std)"] = met4
print(f"  Velocity Smoothness        : {met4:.4f} px/s  (lower = smoother)")

# ─────────────────────────────────────────────────────────────
# Metric 5: Mean Posterior Confidence + Fusion Gain
# ─────────────────────────────────────────────────────────────
if not master_df.empty and not detections_df.empty:
    met5_post  = round(float(master_df["confidence"].mean()), 4)
    met5_prior = round(float(detections_df["confidence"].mean()), 4)
    met5_gain  = round(met5_post - met5_prior, 4)
else:
    met5_post = met5_prior = met5_gain = 0.0
results["5. Mean Posterior Confidence"] = met5_post
results["5. YOLO Prior Confidence"]     = met5_prior
results["5. Fusion Gain"]               = met5_gain
print(f"  Mean Posterior Confidence  : {met5_post:.4f}  "
      f"| Prior: {met5_prior:.4f}  | Gain: {met5_gain:+.4f}")

# ─────────────────────────────────────────────────────────────
# Metric 6: Siren-Emergency Co-occurrence
# ─────────────────────────────────────────────────────────────
if (not master_df.empty
        and "siren_prob"   in master_df.columns
        and "is_emergency" in master_df.columns):
    siren_rows = master_df[master_df["siren_prob"] > 0.5]
    met6 = round(float(siren_rows["is_emergency"].mean()), 4) if len(siren_rows) > 0 else None
else:
    met6 = None
results["6. Siren-Emergency Co-occurrence"] = met6
if met6 is not None:
    print(f"  Siren-Emergency Co-occur   : {met6:.4f}  (P(emergency | siren_prob>0.5))")
else:
    print(f"  Siren-Emergency Co-occur   : N/A  (no siren detections)")

# ─────────────────────────────────────────────────────────────
# Metric 7: Predicate Diversity Entropy
# ─────────────────────────────────────────────────────────────
if not relations_df.empty:
    probs    = relations_df["predicate"].value_counts(normalize=True).values
    met7_raw = round(float(-np.sum(probs * np.log(probs + 1e-12))), 4)
    n_preds  = len(probs)
    met7_norm = round(met7_raw / math.log(n_preds + 1e-12), 4) if n_preds > 1 else 0.0
else:
    met7_raw = met7_norm = 0.0          
results["7. Predicate Diversity Entropy (raw)"]        = met7_raw
results["7. Predicate Diversity Entropy (normalised)"] = met7_norm
print(f"  Predicate Diversity Entropy: {met7_raw:.4f}  "
      f"| Normalised: {met7_norm:.4f}")

# ─────────────────────────────────────────────────────────────
# Metric 8: Node Coverage Ratio
# ─────────────────────────────────────────────────────────────
unique_tracks = scene_stats.get("unique_tracks", 0)
kg_track_ids = set()
for n, d in KG.nodes(data=True):
    if d.get("type") == "object":
        label    = str(d.get("label", ""))
        track_id = label.split("@")[0].strip()
        if track_id:
            kg_track_ids.add(track_id)
kg_unique_tracks = len(kg_track_ids)
met8 = round(kg_unique_tracks / unique_tracks, 4) if unique_tracks > 0 else 0.0
results["8. Node Coverage Ratio"] = met8
print(f"  Node Coverage Ratio        : {met8:.4f}  "
      f"({kg_unique_tracks} unique tracks in KG / {unique_tracks} M1 tracks)")

# ─────────────────────────────────────────────────────────────
# Metric 9: Mean Edge Confidence
# ─────────────────────────────────────────────────────────────
edge_confs = [d.get("confidence", 0.0) for _, _, d in KG.edges(data=True)]
met9 = round(float(np.mean(edge_confs)), 4) if edge_confs else 0.0
results["9. Mean Edge Confidence (KG)"] = met9
print(f"  Mean Edge Confidence (KG)  : {met9:.4f}  ({len(edge_confs)} edges)")

# ─────────────────────────────────────────────────────────────
# Metric 10: Prune Efficiency
# ─────────────────────────────────────────────────────────────
PRUNE_AGE  = 5.0
video_end  = video_meta.duration
pruned_count = surviving_count = 0
for u, v, d in KG.edges(data=True):
    age_at_end = video_end - d.get("last_seen", 0.0)
    if age_at_end > PRUNE_AGE:
        pruned_count += 1
    else:
        surviving_count += 1
total_edges = pruned_count + surviving_count
met10 = round(pruned_count / total_edges, 4) if total_edges > 0 else 0.0
results["10. Prune Efficiency"] = met10
print(f"  Prune Efficiency           : {met10:.4f}  "
      f"({pruned_count} stale / {total_edges} total edges at video end)")

# ─────────────────────────────────────────────────────────────
# Metric 11: Event Diversity Score
# ─────────────────────────────────────────────────────────────
TOTAL_DETECTORS = 13
if not events_df.empty:
    met11 = round(events_df["event_type"].nunique() / TOTAL_DETECTORS, 4)
    fired = events_df["event_type"].unique().tolist()
else:
    met11 = 0.0; fired = []
results["11. Event Diversity Score"] = met11
print(f"  Event Diversity Score      : {met11:.4f}  "
      f"({len(fired)}/{TOTAL_DETECTORS} detectors fired)")
if fired:
    print(f"       Fired: {fired}")

# ─────────────────────────────────────────────────────────────
# Metric 12: AV vs Vision-Only Event Delta
# ─────────────────────────────────────────────────────────────
print(f"\n  Running Vision-Only baseline (reused for Metric 17)...")
_m5_vo       = M5EventEngine()
events_vo_df = _m5_vo.run_all(master_df, relations_df, [],
                               frame_height=video_meta.height)

av_count = len(events_df)    if not events_df.empty    else 0
vo_count = len(events_vo_df) if not events_vo_df.empty else 0
met12_delta = av_count - vo_count
results["12. AV Event Count"]          = av_count
results["12. Vision-Only Event Count"] = vo_count
results["12. AV vs VO Event Delta"]    = met12_delta
print(f"  AV vs Vision-Only Delta    : {met12_delta:+d}  "
      f"(AV={av_count}  |  VO={vo_count})")

all_types = set()
if not events_df.empty:    all_types |= set(events_df["event_type"].unique())
if not events_vo_df.empty: all_types |= set(events_vo_df["event_type"].unique())
for et in sorted(all_types):
    av_n = len(events_df[events_df["event_type"]==et])       if not events_df.empty    else 0
    vo_n = len(events_vo_df[events_vo_df["event_type"]==et]) if not events_vo_df.empty else 0
    print(f"         {et:<30} AV={av_n}  VO={vo_n}  Delta={av_n-vo_n:+d}")

# ─────────────────────────────────────────────────────────────
# Metric 13: Entity Coverage
# ─────────────────────────────────────────────────────────────
kg_classes = set()
for n, d in KG.nodes(data=True):
    if d.get("type") == "object":
        label      = str(d.get("label", "")).lower()
        track_part = label.split("@")[0]
        class_part = re.split(r"[_\d]", track_part)[0]
        if class_part:
            kg_classes.add(class_part)
narrative_lower = narrative.lower() if narrative else ""
covered = [c for c in kg_classes if c in narrative_lower]
met13 = round(len(covered) / len(kg_classes), 4) if kg_classes else 0.0
results["13. Entity Coverage"] = met13
print(f"\n  Entity Coverage            : {met13:.4f}  "
      f"({len(covered)}/{len(kg_classes)} entity classes in narrative)")
print(f"       Covered : {sorted(covered)}")
print(f"       Missing : {sorted(set(kg_classes) - set(covered))}")

# ─────────────────────────────────────────────────────────────
# Metric 14: Entity Hallucination Rate
# ─────────────────────────────────────────────────────────────
valid_classes = set()
if not master_df.empty:
    for cls in master_df["class_name"].dropna().unique():
        valid_classes.add(cls.lower().strip())
for n, d in KG.nodes(data=True):
    label      = str(d.get("label", "")).lower()
    track_part = label.split("@")[0]
    class_part = re.split(r"[_\d]", track_part)[0]
    if class_part:
        valid_classes.add(class_part)

urban_vocab = {"car","truck","bus","ambulance","police","motorcycle",
               "bicycle","person","pedestrian","vehicle","fire","smoke",
               "crowd","emergency","siren","traffic","collision"}

# Regex finds urban-vocab words in narrative; stopwords are excluded by urban_vocab filter
narrative_entities    = set(re.findall(r'\b[a-z]+\b', narrative_lower)) & urban_vocab
hallucinated_entities = [e for e in narrative_entities
                         if e not in valid_classes
                         and e not in {"vehicle","traffic","emergency","siren",
                                       "pedestrian","collision","crowd","fire","smoke"}]

duration_sec      = video_meta.duration
hallucinated_time = []
for m in re.finditer(r'(\d+(?:\.\d+)?)\s*(minute[s]?|hour[s]?)', narrative_lower):
    val  = float(m.group(1))
    unit = m.group(2)
    secs = val * 60 if "minute" in unit else val * 3600
    if secs > duration_sec * 2:
        hallucinated_time.append(m.group(0))

total_hallucinated = len(hallucinated_entities) + len(hallucinated_time)
total_checked      = len(narrative_entities) + len(hallucinated_time)
met14 = round(total_hallucinated / total_checked, 4) if total_checked > 0 else 0.0
results["14. Entity Hallucination Rate"]        = met14
results["14. Hallucinated Entities"]            = hallucinated_entities
results["14. Hallucinated Time Expressions"]    = hallucinated_time
print(f"  Entity Hallucination Rate  : {met14:.4f}  "
      f"({total_hallucinated} hallucinated / {total_checked} checked)")
if hallucinated_entities:
    print(f"       Hallucinated entities : {hallucinated_entities}")
if hallucinated_time:
    print(f"       Hallucinated time expr: {hallucinated_time}  "
          f"(video is only {duration_sec:.1f}s)")

# ─────────────────────────────────────────────────────────────
# Metric 15: Event Coverage in Narrative
# ─────────────────────────────────────────────────────────────
EVENT_KEYWORDS = {
    "EmergencyVehicleDetected": ["emergency","ambulance","police","fire truck","siren"],
    "Collision":                ["collision","crash","collid"],
    "NearMiss":                 ["near miss","nearmiss","close call"],
    "Speeding":                 ["speed","fast","rapid"],
    "StalledVehicle":           ["stall","stopped","stationary"],
    "TrafficCongestion":        ["congestion","congested","traffic","slow"],
    "HonkResponse":             ["honk","horn"],
    "PedestrianInRoad":         ["pedestrian","person","walking"],
    "WrongWayDriving":          ["wrong way","wrong-way","opposite"],
    "SuddenBraking":            ["brake","braking","decelerat"],
    "RunningRedLight":          ["red light","signal"],
    "FireOrSmoke":              ["fire","smoke"],
    "CrowdDisturbance":         ["crowd","disturbance","gathering"],
}
if not events_df.empty:
    fired_types   = events_df["event_type"].unique().tolist()
    event_covered = [et for et in fired_types
                     if any(kw in narrative_lower
                            for kw in EVENT_KEYWORDS.get(et, [et.lower()]))]
    met15 = round(len(event_covered) / len(fired_types), 4) if fired_types else 0.0
else:
    fired_types = event_covered = []
    met15 = 0.0
results["15. Event Coverage in Narrative"] = met15
print(f"  Event Coverage in Narrative: {met15:.4f}  "
      f"({len(event_covered)}/{len(fired_types)} events reflected)")

# ─────────────────────────────────────────────────────────────
# Metric 16: Narrative Length Check
# ─────────────────────────────────────────────────────────────
sentences   = [s.strip() for s in re.split(r'[.!?]', narrative)
               if s.strip()] if narrative else []
met16_count = len(sentences)
met16_pass  = "PASS" if 3 <= met16_count <= 5 else "ACCEPTABLE" if met16_count == 6 else "FAIL"
results["16. Narrative Sentence Count"] = met16_count
results["16. Narrative Length Check"]   = met16_pass
print(f"  Narrative Length Check     : {met16_count} sentences → {met16_pass}  (target: 3-5)")

# ─────────────────────────────────────────────────────────────
# Metric 17: Operational Throughput
# ─────────────────────────────────────────────────────────────
import time as _time
met17_approx = round(MAX_FRAMES / video_meta.duration, 2) if video_meta.duration > 0 else 0.0

t0 = _time.time()
_m5_vo.run_all(master_df, relations_df, [], frame_height=video_meta.height)
met5_time     = _time.time() - t0
met5_fps_proxy = round(MAX_FRAMES / met5_time, 2) if met5_time > 0 else 0.0

results["17. Throughput - Content Rate (FPS)"] = met17_approx
results["17. Speed (FPS proxy)"]               = met5_fps_proxy
print(f"  Throughput (content rate)  : {met17_approx:.2f} FPS")
print(f"  Speed (proxy)              : {met5_fps_proxy:.2f} FPS  (timed: {met5_time:.2f}s)")

# ─────────────────────────────────────────────────────────────
# Metric 18: Class Recall
# ─────────────────────────────────────────────────────────────
detected_classes  = set(
    row['class_name'].lower()
    for _, row in master_df.drop_duplicates("track_id").iterrows()
)
mentioned_classes = {c for c in detected_classes if c in narrative_lower}
met18 = round(len(mentioned_classes) / len(detected_classes), 4) if detected_classes else 0.0
results["18. Class Recall"] = met18
print(f"  Class Recall               : {met18:.4f}  "
      f"({len(mentioned_classes)}/{len(detected_classes)} classes mentioned)")
print(f"       Detected : {sorted(detected_classes)}")
print(f"       Missing  : {sorted(detected_classes - mentioned_classes)}")

# ─────────────────────────────────────────────────────────────
# Metric 19: Compression Ratio
# ─────────────────────────────────────────────────────────────
if not active_triplets:
    print("  Compression Ratio          : N/A (no active triplets)")
    results["19. Compression Ratio"] = None
else:
    input_text = " ".join([
        f"{t['subject']} {t['predicate']} {t['object']}"
        for t in active_triplets
    ])
    met19 = round(len(narrative.split()) / max(1, len(input_text.split())), 4)
    results["19. Compression Ratio"] = met19
    print(f"  Compression Ratio          : {met19:.4f}  (target < 0.3)")

# ─────────────────────────────────────────────────────────────
# Metric 20: BERTScore
# ─────────────────────────────────────────────────────────────
if active_triplets and narrative:
    pseudo_ref = " ".join([
        f"{t['subject'].split('@')[0]} {t['predicate']} {t['object'].split('@')[0]}"
        for t in sorted(active_triplets, key=lambda x: -x["confidence"])[:15]
    ])
    P, R, F1 = bert_score_fn([narrative], [pseudo_ref], lang="en", verbose=False)
    results["20. BERTScore Precision"] = round(P.mean().item(), 4)
    results["20. BERTScore Recall"]    = round(R.mean().item(), 4)
    results["20. BERTScore F1"]        = round(F1.mean().item(), 4)
    print(f"  BERTScore Precision        : {P.mean().item():.4f}")
    print(f"  BERTScore Recall           : {R.mean().item():.4f}")
    print(f"  BERTScore F1               : {F1.mean().item():.4f}")
else:
    print("  BERTScore                  : skipped (no triplets or empty narrative)")
    results["20. BERTScore Precision"] = None
    results["20. BERTScore Recall"]    = None
    results["20. BERTScore F1"]        = None

# ─────────────────────────────────────────────────────────────
# Final Scorecard Print
# ─────────────────────────────────────────────────────────────
print("\n" + "╔" + "═"*62 + "╗")
print("║  FINAL EVALUATION SCORECARD" + " "*34 + "║")
print("╠" + "═"*62 + "╣")
print(f"  {'Metric':<46} {'Value':>10}")
print("─"*62)
for k, v in results.items():
    if isinstance(v, list):
        continue
    val_str = ("N/A"       if v is None
               else f"{v:.4f}" if isinstance(v, float)
               else f"{v:>5d}" if isinstance(v, int)
               else str(v))
    print(f"  {k:<46} {val_str:>10}")
print("╚" + "═"*62 + "╝")

# ─────────────────────────────────────────────────────────────
# Save everything to JSON (all 20 metrics included)
# ─────────────────────────────────────────────────────────────
import json, os

eval_report = {}
for k, v in results.items():
    if isinstance(v, list):
        eval_report[k] = v
    elif v is None:
        eval_report[k] = "N/A"
    else:
        eval_report[k] = v

eval_path = os.path.join(WORKING_DIR, "evaluation_scorecard.json")
with open(eval_path, "w") as f:
    json.dump(eval_report, f, indent=2)
print(f"\n[Eval] Scorecard saved → {eval_path}")
print(f"[Eval] Total metrics saved: {len(eval_report)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 19: Pipeline Integrity Check                   ║
# ╚══════════════════════════════════════════════════════════╝

assert isinstance(m0, M0Preprocessor),       f"m0 wrong type: {type(m0)}"
assert isinstance(m1, M1Perception),          f"m1 wrong type: {type(m1)}"
assert isinstance(m2, M2Tracking),            f"m2 wrong type: {type(m2)}"
assert isinstance(m3, M3RelationInference),   f"m3 wrong type: {type(m3)}"
assert isinstance(m4, M4DynamicKnowledgeGraph), f"m4 wrong type: {type(m4)}"
assert isinstance(m5, M5EventEngine),         f"m5 wrong type: {type(m5)}"
assert isinstance(m6, M6Summarizer),          f"m6 wrong type: {type(m6)}"
assert isinstance(av_fusion, BayesianAVFusion), f"av_fusion wrong type: {type(av_fusion)}"

print("[Integrity] All module instances verified.")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  Cell 20: Batch Processing                           ║
# ╚══════════════════════════════════════════════════════════╝
from pathlib import Path
import pandas as pd

def run_pipeline_on_video(video_path: str) -> dict:
    """Run the full M0→M8 pipeline on a single video. Returns report dict."""
    
    out_path = reencode_video(video_path, WORKING_DIR)
    
    # M0
    m0_out        = m0.run(out_path)
    v_meta        = m0_out["metadata"]
    a_tokens      = m0_out["audio_tokens"]
    p_mode        = m0_out["mode"]
    #global pipeline_mode
    #pipeline_mode = p_mode
    # pipeline_mode is local to this call — returned in the result dict
    
    # M1
    m1_out        = m1.detect(out_path, a_tokens, fps_override=v_meta.fps)
    dets_df       = m1_out["detections_df"]
    fps_v         = m1_out["fps"]
    s_stats       = m1_out["scene_stats"]
    
    # Dedup
    dets_df = deduplicate_and_reidentify(dets_df, fps=fps_v)
    s_stats["unique_tracks"] = int(dets_df["track_id"].nunique())
    
    # M2 → M3 → M4(KG) → M5 → M6
    tr_df   = m2.track(dets_df, fps=fps_v)
    mst_df  = av_fusion.fuse(tr_df, a_tokens)
    rel_df  = m3.infer_triplets(mst_df, a_tokens, fps=fps_v)
    kg      = m4.build_from_relations(mst_df, rel_df, a_tokens, fps=fps_v)
    ev_df   = m5.run_all(mst_df, rel_df, a_tokens,
                         frame_height=v_meta.height, frame_width=v_meta.width)
    narr    = m6.generate(m4, ev_df, s_stats, p_mode,
                          duration_sec=v_meta.duration)
    
    return {
        "video":       Path(video_path).name,
        "events":      ev_df,
        "narrative":   narr,
        "scene_stats": s_stats,
        "meta":        v_meta,
        "mode": p_mode
    }


# ── Batch runner ──────────────────────────────────────────────────────────────
VIDEO_DIR    = "/kaggle/input/datasets/dharshini25v/batch-processing"
video_files  = sorted(Path(VIDEO_DIR).glob("*.mp4"))

if not video_files:
    print("[BATCH] No .mp4 files found in VIDEO_DIR — check the path.")

batch_results = []
for vp in video_files:
    print(f"\n{'='*60}\nProcessing: {vp.name}\n{'='*60}")
    try:
        result = run_pipeline_on_video(str(vp))
        batch_results.append(result)
    except Exception as e:
        import traceback
        print(f"[BATCH] Failed on {vp.name}: {e}")
        traceback.print_exc()          


# ── Aggregate event summary ───────────────────────────────────────────────────
valid = [
    r["events"].assign(video=r["video"])
    for r in batch_results
    if r["events"] is not None and not r["events"].empty
]

if valid:
    all_events = pd.concat(valid, ignore_index=True)
    print("\n=== EVENT SUMMARY ===")
    print(all_events.groupby(["video", "event_type"]).size().to_string())
else:
    print("[BATCH] No events detected across any video.")


# ── Narratives ────────────────────────────────────────────────────────────────
print("\n=== NARRATIVES ===")
for r in batch_results:                         
    print(f"\n--- {r['video']} ---")

    if r["events"] is None or r["events"].empty:
        print("  [Skipped — no events]")
        continue                                 

    narr = r.get("narrative")
    if narr is None:
        print("  [No narrative generated]")
    elif isinstance(narr, str):
        print(narr)
    elif isinstance(narr, dict):
        for k, v in narr.items():
            print(f"  [{k}]: {v}")
    else:
        print(narr)